## **Data Exploration**

In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath('..')
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'raw', 'Superstore sales dataset.csv'), encoding='utf-8')

print('='*60)
print('DATASET SHAPE')
print('='*60)
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

print('\n' + '='*60)
print('COLUMN NAMES AND DTYPES')
print('='*60)
for col in df.columns:
    print(f'  {col:20s} -> {df[col].dtype}')

print('\n' + '='*60)
print('MISSING VALUES PER COLUMN')
print('='*60)
missing = df.isnull().sum()
for col in df.columns:
    pct = (missing[col] / len(df)) * 100
    print(f'  {col:20s} -> {missing[col]:5d} missing ({pct:.2f}%)')
print(f'\n  TOTAL missing cells: {missing.sum()}')

print('\n' + '='*60)
print('UNIQUE VALUES PER COLUMN')
print('='*60)
for col in df.columns:
    print(f'  {col:20s} -> {df[col].nunique():6d} unique')

print('\n' + '='*60)
print('NUMERIC COLUMNS SUMMARY')
print('='*60)
print(df.describe().to_string())

print('\n' + '='*60)
print('DATE COLUMNS ANALYSIS')
print('='*60)
print('Order Date sample values:', df['Order Date'].head(10).tolist())
print('Ship Date sample values:', df['Ship Date'].head(10).tolist())

order_dates = pd.to_datetime(df['Order Date'], format='%d/%m/%Y', errors='coerce')
ship_dates = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y', errors='coerce')
print(f'Order Date - parsed (d/m/Y): {order_dates.notna().sum()} of {len(df)}')
print(f'  Date range: {order_dates.min()} to {order_dates.max()}')
print(f'Ship Date - parsed (d/m/Y): {ship_dates.notna().sum()} of {len(df)}')
print(f'  Date range: {ship_dates.min()} to {ship_dates.max()}')

# Check mixed formats
order_dates_us = pd.to_datetime(df['Order Date'], format='%m/%d/%Y', errors='coerce')
print(f'Order Date - parsed (m/d/Y): {order_dates_us.notna().sum()} of {len(df)}')

shipping_days = (ship_dates - order_dates).dt.days
print(f'\nShipping days stats:')
print(f'  Min: {shipping_days.min()}, Max: {shipping_days.max()}, Mean: {shipping_days.mean():.1f}')
print(f'  Negative shipping days: {(shipping_days < 0).sum()}')

print('\n' + '='*60)
print('CATEGORICAL COLUMNS BREAKDOWN')
print('='*60)
cat_cols = ['Ship Mode', 'Segment', 'Country', 'Region', 'Category', 'Sub-Category']
for col in cat_cols:
    print(f'\n{col} ({df[col].nunique()} unique):')
    vc = df[col].value_counts()
    for val, cnt in vc.items():
        print(f'  {str(val):30s} -> {cnt:5d} ({cnt/len(df)*100:.1f}%)')

print('\n' + '='*60)
print('STATE DISTRIBUTION (Top 15)')
print('='*60)
state_vc = df['State'].value_counts()
for val, cnt in state_vc.head(15).items():
    print(f'  {val:30s} -> {cnt:5d} ({cnt/len(df)*100:.1f}%)')
total_states = df['State'].nunique()
print(f'  Total unique states: {total_states}')

print('\n' + '='*60)
print('DUPLICATE ROWS')
print('='*60)
dupes = df.duplicated().sum()
print(f'  Exact duplicate rows: {dupes}')
dupes_subset = df.duplicated(subset=['Order ID', 'Product ID']).sum()
print(f'  Duplicate Order ID + Product ID combos: {dupes_subset}')

print('\n' + '='*60)
print('NEGATIVE PROFIT ROWS')
print('='*60)
neg_profit = df[df['Profit'] < 0]
print(f'  Rows with negative profit: {len(neg_profit)} ({len(neg_profit)/len(df)*100:.1f}%)')
print(f'  Total loss amount: ${neg_profit["Profit"].sum():,.2f}')

print('\n' + '='*60)
print('DISCOUNT ANALYSIS')
print('='*60)
zero_disc = (df['Discount'] == 0).sum()
print(f'  Zero discount rows: {zero_disc} ({zero_disc/len(df)*100:.1f}%)')
print(f'  Max discount: {df["Discount"].max()}')
print(f'  Mean discount: {df["Discount"].mean():.3f}')

print('\n' + '='*60)
print('POSTAL CODE ANALYSIS')
print('='*60)
pc_missing = df['Postal Code'].isnull().sum()
pc_unique = df['Postal Code'].nunique()
print(f'  Missing postal codes: {pc_missing}')
print(f'  Unique postal codes: {pc_unique}')

print('\n' + '='*60)
print('OUTLIER CHECK (Sales & Profit)')
print('='*60)
q1_sales = df['Sales'].quantile(0.25)
q3_sales = df['Sales'].quantile(0.75)
iqr_sales = q3_sales - q1_sales
outliers_sales = ((df['Sales'] < q1_sales - 1.5*iqr_sales) | (df['Sales'] > q3_sales + 1.5*iqr_sales)).sum()
print(f'  Sales IQR: {iqr_sales:.2f}, Outliers (1.5*IQR): {outliers_sales}')

q1_profit = df['Profit'].quantile(0.25)
q3_profit = df['Profit'].quantile(0.75)
iqr_profit = q3_profit - q1_profit
outliers_profit = ((df['Profit'] < q1_profit - 1.5*iqr_profit) | (df['Profit'] > q3_profit + 1.5*iqr_profit)).sum()
print(f'  Profit IQR: {iqr_profit:.2f}, Outliers (1.5*IQR): {outliers_profit}')

# Skewness
print(f'  Sales skewness: {df["Sales"].skew():.3f}')
print(f'  Profit skewness: {df["Profit"].skew():.3f}')


## **Data Cleaning**

In [ ]:
# Standard Library
import os
import sys
import logging
import warnings
from datetime import datetime
from io import StringIO

# Third-Party
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Paths relative to project root
PROJECT_ROOT = os.path.abspath('..')
RAW_DATA       = os.path.join(PROJECT_ROOT, 'data', 'raw', 'Superstore sales dataset.csv')
CLEAN_DATA     = os.path.join(PROJECT_ROOT, 'data', 'processed', 'clean_data.csv')
LOG_FILE       = os.path.join(PROJECT_ROOT, 'logs', 'cleaning_log.txt')
SUMMARY_FILE   = os.path.join(PROJECT_ROOT, 'logs', 'cleaning_summary.txt')

# Logging Setup
logger = logging.getLogger("DataCleaner")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
fmt = logging.Formatter("%(asctime)s  [%(levelname)-8s]  %(message)s",
                        datefmt="%Y-%m-%d %H:%M:%S")
file_handler.setFormatter(fmt)
logger.addHandler(file_handler)

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)
console.setFormatter(fmt)
logger.addHandler(console)


# Section banner helper
def section(title: str):
    logger.info("-" * 70)
    logger.info(f"  {title}")
    logger.info("-" * 70)


# Step 1: Load raw data and take a snapshot for comparison
section("1. Loading Raw Data")

df_raw = pd.read_csv(RAW_DATA, encoding="utf-8")
df = df_raw.copy()

ORIG_ROWS, ORIG_COLS = df.shape
logger.info(f"Loaded {ORIG_ROWS:,} rows × {ORIG_COLS} columns from '{RAW_DATA}'")
logger.info(f"Columns: {list(df.columns)}")
logger.info(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Snapshot for final comparison
orig_dtypes   = df.dtypes.copy()
orig_describe = df.describe().copy()
orig_missing  = df.isnull().sum().copy()


# Step 2: Remove exact duplicate rows
section("2. Exact Duplicate Removal")

exact_dupes = df.duplicated().sum()
logger.info(f"Exact duplicate rows detected: {exact_dupes}")

if exact_dupes > 0:
    dupe_mask = df.duplicated(keep="first")
    logger.debug(f"Row indices of duplicates: {df[dupe_mask].index.tolist()}")
    df.drop_duplicates(keep="first", inplace=True)
    logger.info(f"Removed {exact_dupes} exact duplicate row(s).  Shape → {df.shape}")
else:
    logger.info("No exact duplicates found. Data unchanged.")


# Step 3: Remove partial duplicates based on Order ID + Product ID
section("3. Partial Duplicate Removal (Order ID + Product ID)")

partial_dupes = df.duplicated(subset=["Order ID", "Product ID"], keep=False)
n_partial = partial_dupes.sum()
logger.info(f"Rows involved in duplicate (Order ID + Product ID) pairs: {n_partial}")

if n_partial > 0:
    dup_groups = df[partial_dupes].groupby(["Order ID", "Product ID"])
    for (oid, pid), grp in dup_groups:
        logger.debug(f"  Order={oid}, Product={pid} → {len(grp)} rows "
                     f"(keeping first, indices {grp.index.tolist()})")

    before = len(df)
    df.drop_duplicates(subset=["Order ID", "Product ID"], keep="first", inplace=True)
    removed = before - len(df)
    logger.info(f"Removed {removed} partial duplicate row(s).  Shape → {df.shape}")
else:
    logger.info("No partial duplicates found. Data unchanged.")


# Step 4: Analyze and handle missing values
section("4. Missing Value Analysis & Imputation")

missing_per_col = df.isnull().sum()
total_missing   = missing_per_col.sum()
logger.info(f"Total missing cells: {total_missing} "
            f"({total_missing / df.size * 100:.2f}% of all cells)")

if total_missing > 0:
    for col in df.columns:
        n_miss = missing_per_col[col]
        if n_miss == 0:
            continue
        pct = n_miss / len(df) * 100
        logger.info(f"  {col:20s} → {n_miss:5d} missing ({pct:.2f}%)")

    # Numeric columns: median imputation (robust to outliers)
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if df[col].isnull().any():
            med = df[col].median()
            df[col].fillna(med, inplace=True)
            logger.info(f"  Imputed '{col}' with median = {med:.4f}")

    # Categorical columns: mode imputation
    cat_cols = df.select_dtypes(include=["object"]).columns
    for col in cat_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            logger.info(f"  Imputed '{col}' with mode = '{mode_val}'")

    # Forward-fill / Backward-fill for sequential columns
    # Row ID should be sequential; fill gaps if any
    if df["Row ID"].isnull().any():
        df["Row ID"].ffill(inplace=True)
        df["Row ID"].bfill(inplace=True)
        logger.info("  Forward/backward-filled 'Row ID'")
else:
    logger.info("Dataset is complete — zero missing values. No imputation required.")

# Step 5: Parse, standardize, and validate date columns
section("5. Date Parsing, Standardization & Validation")

# Detect dominant date format
logger.info("Detecting date format …")

# Try D/M/YYYY first
parsed_dm  = pd.to_datetime(df["Order Date"], format="%d/%m/%Y", errors="coerce")
# Try M/D/YYYY
parsed_md  = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")

dm_ok  = parsed_dm.notna().sum()
md_ok  = parsed_md.notna().sum()
logger.info(f"  Parsed with D/M/YYYY: {dm_ok:,} / {len(df):,}")
logger.info(f"  Parsed with M/D/YYYY: {md_ok:,} / {len(df):,}")

if dm_ok >= md_ok:
    chosen_fmt  = "%d/%m/%Y"
    chosen_label = "D/M/YYYY"
else:
    chosen_fmt  = "%m/%d/%Y"
    chosen_label = "M/D/YYYY"

logger.info(f"  ► Chosen format: {chosen_label} (higher parse success)")

# Convert to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"], format=chosen_fmt, errors="coerce")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format=chosen_fmt, errors="coerce")

unparsed_order = df["Order Date"].isnull().sum()
unparsed_ship  = df["Ship Date"].isnull().sum()
logger.info(f"  Unparseable Order Dates: {unparsed_order}")
logger.info(f"  Unparseable Ship Dates:  {unparsed_ship}")

# Validate date ranges
order_min, order_max = df["Order Date"].min(), df["Order Date"].max()
ship_min,  ship_max  = df["Ship Date"].min(),  df["Ship Date"].max()
logger.info(f"  Order Date range: {order_min.date()} → {order_max.date()}")
logger.info(f"  Ship Date range:  {ship_min.date()}  → {ship_max.date()}")

# Flag future dates (beyond current year + 1 as safety margin)
future_cutoff = pd.Timestamp(datetime.now().year + 1, 1, 1)
future_orders = (df["Order Date"] > future_cutoff).sum()
future_ships  = (df["Ship Date"]  > future_cutoff).sum()
logger.info(f"  Future-dated orders (>{future_cutoff.year}-01-01): {future_orders}")
logger.info(f"  Future-dated shipments: {future_ships}")

# Shipping duration validation
df["Shipping_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

neg_ship = (df["Shipping_Days"] < 0).sum()
logger.info(f"  Rows where Ship Date < Order Date (negative days): {neg_ship}")

if neg_ship > 0:
    logger.info(f"  ► Removing {neg_ship} invalid shipping rows")
    df = df[df["Shipping_Days"] >= 0].copy()
    logger.info(f"  Shape after removal → {df.shape}")

extreme_ship = (df["Shipping_Days"] > 365).sum()
logger.info(f"  Rows with shipping > 365 days (outlier): {extreme_ship}")

logger.info(f"  Shipping days — Min: {df['Shipping_Days'].min()}, "
            f"Max: {df['Shipping_Days'].max()}, "
            f"Mean: {df['Shipping_Days'].mean():.1f}, "
            f"Median: {df['Shipping_Days'].median():.0f}")


# Step 6: Remove rows with impossible values
section("6. Invalid Row Detection & Removal")

invalid_count = 0

# Negative or zero Quantity
neg_qty = (df["Quantity"] <= 0).sum()
logger.info(f"  Rows with Quantity ≤ 0: {neg_qty}")
if neg_qty > 0:
    df = df[df["Quantity"] > 0].copy()
    invalid_count += neg_qty
    logger.info(f"  ► Removed {neg_qty} row(s).  Shape → {df.shape}")

# Negative Sales (impossible for revenue)
neg_sales = (df["Sales"] < 0).sum()
logger.info(f"  Rows with negative Sales: {neg_sales}")
if neg_sales > 0:
    df = df[df["Sales"] >= 0].copy()
    invalid_count += neg_sales
    logger.info(f"  ► Removed {neg_sales} row(s).  Shape → {df.shape}")

# Quantity > 100 (unrealistic for retail)
extreme_qty = (df["Quantity"] > 100).sum()
logger.info(f"  Rows with Quantity > 100: {extreme_qty}")
if extreme_qty > 0:
    df = df[df["Quantity"] <= 100].copy()
    invalid_count += extreme_qty
    logger.info(f"  ► Removed {extreme_qty} row(s).  Shape → {df.shape}")

# Discount > 1.0 (> 100% is invalid)
over_disc = (df["Discount"] > 1.0).sum()
logger.info(f"  Rows with Discount > 100%: {over_disc}")
if over_disc > 0:
    df = df[df["Discount"] <= 1.0].copy()
    invalid_count += over_disc
    logger.info(f"  ► Removed {over_disc} row(s).  Shape → {df.shape}")

# (zero revenue — likely test/cancelled orders)
zero_sales = ((df["Sales"] == 0) & (df["Quantity"] > 0)).sum()
logger.info(f"  Rows with Sales = 0 but Quantity > 0: {zero_sales}")

logger.info(f"  Total invalid rows removed in Step 6: {invalid_count}")


# Step 7: Validate categorical columns against known valid values
section("7. Categorical Data Validation")

# Ship Mode
valid_ship_modes = {"Standard Class", "Second Class", "First Class", "Same Day"}
invalid_modes = df[~df["Ship Mode"].isin(valid_ship_modes)]["Ship Mode"].unique()
logger.info(f"  Ship Mode — valid values: {valid_ship_modes}")
logger.info(f"  Ship Mode — unexpected values: {invalid_modes.tolist() if len(invalid_modes) else 'None'}")

# Segment
valid_segments = {"Consumer", "Corporate", "Home Office"}
invalid_segs = df[~df["Segment"].isin(valid_segments)]["Segment"].unique()
logger.info(f"  Segment — valid values: {valid_segments}")
logger.info(f"  Segment — unexpected values: {invalid_segs.tolist() if len(invalid_segs) else 'None'}")

# Region
valid_regions = {"West", "East", "Central", "South"}
invalid_regs = df[~df["Region"].isin(valid_regions)]["Region"].unique()
logger.info(f"  Region — valid values: {valid_regions}")
logger.info(f"  Region — unexpected values: {invalid_regs.tolist() if len(invalid_regs) else 'None'}")

# Category
valid_categories = {"Office Supplies", "Furniture", "Technology"}
invalid_cats = df[~df["Category"].isin(valid_categories)]["Category"].unique()
logger.info(f"  Category — valid values: {valid_categories}")
logger.info(f"  Category — unexpected values: {invalid_cats.tolist() if len(invalid_cats) else 'None'}")

# Country (should be single value)
countries = df["Country"].unique()
logger.info(f"  Country unique values: {countries.tolist()}")
if len(countries) > 1:
    logger.warning("  Multiple countries detected — verify data scope")

# Sub-Category consistency with Category
expected_mapping = {
    "Office Supplies": {"Binders", "Paper", "Furnishings", "Storage", "Art",
                        "Appliances", "Labels", "Envelopes", "Fasteners", "Supplies"},
    "Furniture":       {"Chairs", "Tables", "Bookcases", "Furnishings"},
    "Technology":      {"Phones", "Accessories", "Machines", "Copiers"},
}

cross_check = df[["Category", "Sub-Category"]].drop_duplicates()
for _, row in cross_check.iterrows():
    cat, subcat = row["Category"], row["Sub-Category"]
    if cat in expected_mapping:
        if subcat not in expected_mapping[cat]:
            # Furnishings appears in both Furniture and Office Supplies — known
            if not (cat == "Furniture" and subcat == "Furnishings"):
                logger.warning(f"  Unexpected mapping: {cat} → {subcat}")

logger.info("  Sub-Category ↔ Category cross-validation complete.")

# Whitespace & casing cleanup
str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    before = df[col].copy()
    df[col] = df[col].astype(str).str.strip()
    changed = (before != df[col]).sum()
    if changed > 0:
        logger.info(f"  Stripped whitespace in '{col}': {changed} value(s) changed")

logger.info("  Categorical validation complete.")


# Step 8: Detect outliers using IQR, Z-Score, Modified Z-Score, and Isolation Forest
section("8. Advanced Outlier Detection")

# IQR Method (Sales & Profit)
logger.info("── 8a. IQR Method ──")

for col in ["Sales", "Profit"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    logger.info(f"  {col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
                f"Bounds=[{lower:.2f}, {upper:.2f}], Outliers={outliers}")

# Z-Score Method
logger.info("── 8b. Z-Score Method (|z| > 3) ──")

for col in ["Sales", "Profit", "Quantity"]:
    z = np.abs(stats.zscore(df[col], nan_policy="omit"))
    n_outliers = (z > 3).sum()
    logger.info(f"  {col}: Z-score outliers (|z|>3) = {n_outliers} "
                f"({n_outliers/len(df)*100:.2f}%)")

# Modified Z-Score (MAD-based, more robust)
logger.info("── 8c. Modified Z-Score (MAD-based, threshold=3.5) ──")

for col in ["Sales", "Profit"]:
    median = df[col].median()
    mad = np.median(np.abs(df[col] - median))
    if mad == 0:
        mad = 1e-10
    mod_z = 0.6745 * (df[col] - median) / mad
    n_outliers = (np.abs(mod_z) > 3.5).sum()
    logger.info(f"  {col}: MAD={mad:.2f}, Modified Z outliers = {n_outliers} "
                f"({n_outliers/len(df)*100:.2f}%)")

# Isolation Forest (ML-based anomaly detection)
logger.info("── 8d. Isolation Forest (ML Anomaly Detection) ──")

iso_features = ["Sales", "Quantity", "Discount", "Profit"]
X_iso = df[iso_features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iso)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,   # expect ~2% anomalies
    random_state=42,
    n_jobs=-1
)
df_iso = df.loc[X_iso.index].copy()
iso_forest.fit(X_scaled)
df_iso["anomaly_score"] = iso_forest.decision_function(X_scaled)
df_iso["is_anomaly"]    = iso_forest.predict(X_scaled)

anomalies = df_iso[df_iso["is_anomaly"] == -1]
logger.info(f"  Features used: {iso_features}")
logger.info(f"  Contamination set: 2%")
logger.info(f"  Anomalies detected: {len(anomalies)} ({len(anomalies)/len(df)*100:.2f}%)")

if len(anomalies) > 0:
    logger.info(f"  Anomaly profile (means):")
    for feat in iso_features:
        logger.info(f"    {feat:12s}: anomaly mean={anomalies[feat].mean():.2f}  "
                    f"vs normal mean={df_iso[df_iso['is_anomaly']==1][feat].mean():.2f}")

# NOTE: We LOG anomalies but do NOT remove them - outliers are legitimate
# business data (large corporate orders, high-discount clearance, etc.)
logger.info("  ► Anomalies logged but preserved (legitimate business extremes).")


# Step 9: Create derived features from existing columns
section("9. Derived Feature Engineering")

# Year, Month, Quarter from Order Date 
df["Order_Year"]    = df["Order Date"].dt.year
df["Order_Month"]   = df["Order Date"].dt.month
df["Order_Quarter"] = df["Order Date"].dt.quarter
df["Order_DayName"] = df["Order Date"].dt.day_name()
logger.info("  Created: Order_Year, Order_Month, Order_Quarter, Order_DayName")

# Profit Margin
df["Profit_Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0.0
)
logger.info("  Created: Profit_Margin (%)")

# Is_Loss flag
df["Is_Loss"] = (df["Profit"] < 0).astype(int)
logger.info(f"  Created: Is_Loss  (flagged {df['Is_Loss'].sum():,} loss rows)")

# Discount Tier
bins   = [-0.01, 0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
labels = ["No Discount", "Low (1-10%)", "Medium (11-20%)", "High (21-30%)",
          "Very High (31-50%)", "Extreme (51-80%)", "Clearance (81-100%)"]
df["Discount_Tier"] = pd.cut(df["Discount"], bins=bins, labels=labels,
                              include_lowest=True)
logger.info("  Created: Discount_Tier (categorical)")

# Shipping duration already in Shipping_Days
logger.info(f"  Shipping_Days already computed (Step 5d)")


# Step 10: Optimize data types for memory efficiency
section("10. Data Type Optimization")

# Convert appropriate string columns to Categorical for memory efficiency
cat_convert = ["Ship Mode", "Segment", "Country", "Region",
               "Category", "Sub-Category", "Discount_Tier", "Order_DayName"]
for col in cat_convert:
    if col in df.columns:
        df[col] = df[col].astype("category")
        logger.info(f"  '{col}' → category  (unique: {df[col].nunique()})")

# Downcast numeric where safe
df["Row ID"]    = pd.to_numeric(df["Row ID"],    downcast="integer")
df["Quantity"]  = pd.to_numeric(df["Quantity"],  downcast="integer")
df["Is_Loss"]   = pd.to_numeric(df["Is_Loss"],   downcast="integer")

logger.info(f"  Memory after optimization: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")


# Step 11: Final integrity checks after all cleaning steps
section("11. Final Data Integrity Checks")

# Re-check missing
final_missing = df.isnull().sum().sum()
logger.info(f"  Total missing cells after cleaning: {final_missing}")

# Re-check duplicates
final_dupes = df.duplicated().sum()
logger.info(f"  Exact duplicate rows after cleaning: {final_dupes}")

# Primary key uniqueness
pk_dupes = df.duplicated(subset=["Row ID"]).sum()
logger.info(f"  Row ID uniqueness violations: {pk_dupes}")

# Data completeness ratio
total_cells = df.size
non_null    = df.notnull().sum().sum()
logger.info(f"  Data completeness: {non_null}/{total_cells} "
            f"({non_null/total_cells*100:.2f}%)")


# Step 12: Reorder columns and save cleaned dataset
section("12. Saving Cleaned Dataset")

# Reorder columns: identifiers first, then temporal, customer, geo, product, financial, derived
id_cols     = ["Row ID", "Order ID"]
date_cols   = ["Order Date", "Ship Date", "Shipping_Days"]
cust_cols   = ["Customer ID", "Customer Name", "Segment"]
geo_cols    = ["Country", "City", "State", "Postal Code", "Region"]
prod_cols   = ["Product ID", "Category", "Sub-Category", "Product Name"]
fin_cols    = ["Sales", "Quantity", "Discount", "Profit"]
ship_cols   = ["Ship Mode"]
deriv_cols  = ["Order_Year", "Order_Month", "Order_Quarter", "Order_DayName",
               "Profit_Margin", "Is_Loss", "Discount_Tier"]

ordered = id_cols + date_cols + ship_cols + cust_cols + geo_cols + prod_cols + fin_cols + deriv_cols
# Include only columns that exist
ordered = [c for c in ordered if c in df.columns]
df = df[ordered]

df.to_csv(CLEAN_DATA, index=False, encoding="utf-8")

CLEAN_ROWS, CLEAN_COLS = df.shape
logger.info(f"Saved {CLEAN_ROWS:,} rows × {CLEAN_COLS} columns → '{CLEAN_DATA}'")
logger.info(f"New columns added: {set(df.columns) - set(df_raw.columns)}")


# Step 13: Generate and save summary report
section("13. Generating Summary Report")

summary_lines = []

def s(text=""):
    summary_lines.append(text)

s("=" * 65)
s("  DATA CLEANING SUMMARY REPORT")
s("  Superstore Sales Dataset")
s(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
s("=" * 65)
s()
s("DIMENSION COMPARISON")
s("-" * 65)
s(f"  {'Metric':<22s} {'Original':<18s} {'Cleaned':<18s}")
s(f"  {'-'*20:<22s} {'-'*16:<18s} {'-'*16:<18s}")
s(f"  {'Rows':<22s} {ORIG_ROWS:<18,} {CLEAN_ROWS:<18,}")
s(f"  {'Columns':<22s} {ORIG_COLS:<18} {CLEAN_COLS:<18}")
rows_removed = ORIG_ROWS - CLEAN_ROWS
s(f"  {'Rows Removed':<22s} {'—':<18s} {rows_removed:<18,}")
s(f"  {'Removal Rate':<22s} {'—':<18s} {rows_removed/ORIG_ROWS*100:<17.2f}%")
s()
s("CLEANING ACTIONS PERFORMED")
s("-" * 65)
s(f"  Exact duplicates removed          :  {exact_dupes}")
partial_removed = max(0, n_partial - df.duplicated(subset=['Order ID','Product ID'], keep='first').sum() - (ORIG_ROWS - len(df)))
s(f"  Partial duplicates removed        :  see log")
s(f"  Invalid rows removed (Step 6)     :  {invalid_count}")
s(f"  Negative shipping days removed    :  {neg_ship}")
s(f"  Missing values imputed            :  {total_missing}")
s(f"  Date format standardized          :  {chosen_label}")
s(f"  Derived columns added             :  {len(set(df.columns)-set(df_raw.columns))}")
s(f"  Data types optimized              :  {len(cat_convert)} columns")
s(f"  ML anomalies detected (logged)    :  {len(anomalies)}")
s()
s("NUMERIC SUMMARY (Cleaned Data)")
s("-" * 65)
s(df[["Sales", "Quantity", "Discount", "Profit", "Profit_Margin", "Shipping_Days"]].describe().to_string())
s()
s("DATA QUALITY SCORECARD")
s("-" * 65)

completeness = non_null / total_cells * 100
uniqueness   = (1 - df.duplicated().sum() / len(df)) * 100
valid_dates  = (df["Order Date"].notna().sum() / len(df)) * 100

s(f"  Completeness    :  {completeness:.2f}%")
s(f"  Uniqueness      :  {uniqueness:.2f}%")
s(f"  Date Validity   :  {valid_dates:.2f}%")
s(f"  Missing Values  :  {df.isnull().sum().sum()}")
s(f"  Duplicate Rows  :  {df.duplicated().sum()}")

summary_text = "\n".join(summary_lines)

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    f.write(summary_text)

logger.info(f"Summary report saved → '{SUMMARY_FILE}'")

# Print summary to console as well
print("\n")
print(summary_text)

logger.info("═══ PIPELINE COMPLETE ═══")


## **Database Import**

In [ ]:
import sys
import os
import csv
import logging
import time
from datetime import datetime
from decimal import Decimal
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.abspath(os.path.join(os.path.abspath('.'), '..')), '.env'))

import psycopg2
from psycopg2 import sql, extras
import pandas as pd

# Paths relative to project root
PROJECT_ROOT = os.path.abspath('..')
DB_NAME     = os.environ.get("DB_NAME", "working_database")
DB_USER     = os.environ.get("DB_USER", "postgres")
DB_PASSWORD = os.environ.get("DB_PASSWORD", "your_password_here")
DB_HOST     = os.environ.get("DB_HOST", "localhost")
DB_PORT     = os.environ.get("DB_PORT", "5432")
CSV_FILE    = os.path.join(PROJECT_ROOT, "data", "processed", "clean_data.csv")
LOG_FILE    = os.path.join(PROJECT_ROOT, "logs", "db_import_log.txt")
BATCH_SIZE  = 2000

# Logging setup
logger = logging.getLogger("DBImporter")
logger.setLevel(logging.DEBUG)

fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setLevel(logging.DEBUG)
fmt = logging.Formatter("%(asctime)s  [%(levelname)-8s]  %(message)s",
                        datefmt="%Y-%m-%d %H:%M:%S")
fh.setFormatter(fmt)
logger.addHandler(fh)

ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)
ch.setFormatter(fmt)
logger.addHandler(ch)


def section(title):
    """Print a section header."""
    logger.info("-" * 70)
    logger.info(f"  {title}")
    logger.info("-" * 70)


# Step 1: Connect to the database
section("1. Database Connection")

conn = psycopg2.connect(
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=DB_PORT
)
conn.autocommit = False
cur = conn.cursor()

cur.execute("SELECT version();")
pg_version = cur.fetchone()[0]
logger.info(f"Connected to: {pg_version}")
logger.info(f"Database: {DB_NAME}  |  Host: {DB_HOST}:{DB_PORT}")


# Step 2: Load CSV into pandas
section("2. Loading CSV Data")

df = pd.read_csv(CSV_FILE, encoding="utf-8")
logger.info(f"Loaded {len(df):,} rows x {len(df.columns)} columns from '{CSV_FILE}'")
logger.info(f"Columns: {list(df.columns)}")

# Type conversions before insert
df["Order Date"]    = pd.to_datetime(df["Order Date"])
df["Ship Date"]     = pd.to_datetime(df["Ship Date"])
df["Shipping_Days"] = df["Shipping_Days"].astype(int)
df["Quantity"]      = df["Quantity"].astype(int)
df["Order_Year"]    = df["Order_Year"].astype(int)
df["Order_Month"]   = df["Order_Month"].astype(int)
df["Order_Quarter"] = df["Order_Quarter"].astype(int)
df["Is_Loss"]       = df["Is_Loss"].astype(int)
df["Postal Code"]   = df["Postal Code"].astype(str).str.zfill(5)

logger.info("Type conversions applied (dates, integers, postal code zero-padding)")


# Step 3: Drop existing objects for a clean slate
section("3. Dropping Existing Objects")

drop_sql = """
DROP TABLE IF EXISTS fact_sales       CASCADE;
DROP TABLE IF EXISTS dim_customer     CASCADE;
DROP TABLE IF EXISTS dim_product      CASCADE;
DROP TABLE IF EXISTS dim_geography    CASCADE;
DROP TABLE IF EXISTS dim_date         CASCADE;
DROP TABLE IF EXISTS dim_ship_mode    CASCADE;
DROP TABLE IF EXISTS sales_data       CASCADE;
"""
cur.execute(drop_sql)
conn.commit()
logger.info("All existing tables dropped (if any)")


# Step 4: Create the flat denormalized table
section("4. Creating Flat Table: sales_data")

flat_table_ddl = """
CREATE TABLE sales_data (
    row_id              INTEGER          NOT NULL,
    order_id            VARCHAR(20)      NOT NULL,
    order_date          DATE             NOT NULL,
    ship_date           DATE             NOT NULL,
    shipping_days       SMALLINT         NOT NULL,
    ship_mode           VARCHAR(20)      NOT NULL,
    customer_id         VARCHAR(20)      NOT NULL,
    customer_name       VARCHAR(100)     NOT NULL,
    segment             VARCHAR(20)      NOT NULL,
    country             VARCHAR(50)      NOT NULL DEFAULT 'United States',
    city                VARCHAR(100)     NOT NULL,
    state               VARCHAR(50)      NOT NULL,
    postal_code         VARCHAR(10)      NOT NULL,
    region              VARCHAR(20)      NOT NULL,
    product_id          VARCHAR(30)      NOT NULL,
    category            VARCHAR(30)      NOT NULL,
    sub_category        VARCHAR(30)      NOT NULL,
    product_name        TEXT             NOT NULL,
    sales               NUMERIC(12,4)    NOT NULL,
    quantity            SMALLINT         NOT NULL,
    discount            NUMERIC(4,2)     NOT NULL,
    profit              NUMERIC(12,4)    NOT NULL,
    order_year          SMALLINT         NOT NULL,
    order_month         SMALLINT         NOT NULL,
    order_quarter       SMALLINT         NOT NULL,
    order_day_name      VARCHAR(10)      NOT NULL,
    profit_margin       NUMERIC(8,4)     NOT NULL,
    is_loss             SMALLINT         NOT NULL,
    discount_tier       VARCHAR(30)      NOT NULL,

    CONSTRAINT pk_sales_data_row_id      PRIMARY KEY (row_id),
    CONSTRAINT chk_sales_positive        CHECK (sales >= 0),
    CONSTRAINT chk_quantity_positive     CHECK (quantity > 0),
    CONSTRAINT chk_discount_range        CHECK (discount >= 0 AND discount <= 1),
    CONSTRAINT chk_shipping_days_valid   CHECK (shipping_days >= 0),
    CONSTRAINT chk_is_loss_binary        CHECK (is_loss IN (0, 1)),
    CONSTRAINT chk_month_range           CHECK (order_month BETWEEN 1 AND 12),
    CONSTRAINT chk_quarter_range         CHECK (order_quarter BETWEEN 1 AND 4),
    CONSTRAINT chk_ship_date_after_order CHECK (ship_date >= order_date)
);

CREATE INDEX idx_sales_order_date    ON sales_data (order_date);
CREATE INDEX idx_sales_order_year    ON sales_data (order_year);
CREATE INDEX idx_sales_region        ON sales_data (region);
CREATE INDEX idx_sales_state         ON sales_data (state);
CREATE INDEX idx_sales_category      ON sales_data (category);
CREATE INDEX idx_sales_sub_category  ON sales_data (sub_category);
CREATE INDEX idx_sales_segment       ON sales_data (segment);
CREATE INDEX idx_sales_ship_mode     ON sales_data (ship_mode);
CREATE INDEX idx_sales_customer_id   ON sales_data (customer_id);
CREATE INDEX idx_sales_product_id    ON sales_data (product_id);
CREATE INDEX idx_sales_is_loss       ON sales_data (is_loss);
CREATE INDEX idx_sales_discount_tier ON sales_data (discount_tier);
CREATE INDEX idx_sales_profit        ON sales_data (profit);
CREATE INDEX idx_sales_year_month    ON sales_data (order_year, order_month);
CREATE INDEX idx_sales_city_state    ON sales_data (city, state);

COMMENT ON TABLE sales_data IS 'Flat denormalized sales table for analytics';
"""

cur.execute(flat_table_ddl)
conn.commit()
logger.info("Table 'sales_data' created with 29 columns, 15 indexes, 8 constraints")


# Step 5: Bulk import data into the flat table
section("5. Importing Data into sales_data")

flat_columns = [
    "row_id", "order_id", "order_date", "ship_date", "shipping_days",
    "ship_mode", "customer_id", "customer_name", "segment", "country",
    "city", "state", "postal_code", "region", "product_id", "category",
    "sub_category", "product_name", "sales", "quantity", "discount",
    "profit", "order_year", "order_month", "order_quarter", "order_day_name",
    "profit_margin", "is_loss", "discount_tier"
]

csv_to_db_col_map = {
    "Row ID": "row_id", "Order ID": "order_id", "Order Date": "order_date",
    "Ship Date": "ship_date", "Shipping_Days": "shipping_days",
    "Ship Mode": "ship_mode", "Customer ID": "customer_id",
    "Customer Name": "customer_name", "Segment": "segment", "Country": "country",
    "City": "city", "State": "state", "Postal Code": "postal_code",
    "Region": "region", "Product ID": "product_id", "Category": "category",
    "Sub-Category": "sub_category", "Product Name": "product_name",
    "Sales": "sales", "Quantity": "quantity", "Discount": "discount",
    "Profit": "profit", "Order_Year": "order_year", "Order_Month": "order_month",
    "Order_Quarter": "order_quarter", "Order_DayName": "order_day_name",
    "Profit_Margin": "profit_margin", "Is_Loss": "is_loss",
    "Discount_Tier": "discount_tier"
}

df_db = df.rename(columns=csv_to_db_col_map)

insert_sql = sql.SQL(
    "INSERT INTO sales_data ({cols}) VALUES ({placeholders})"
).format(
    cols=sql.SQL(", ").join(sql.Identifier(c) for c in flat_columns),
    placeholders=sql.SQL(", ").join(sql.Placeholder() for _ in flat_columns)
)

start_time = time.time()
total_inserted = 0

for batch_start in range(0, len(df_db), BATCH_SIZE):
    batch = df_db.iloc[batch_start:batch_start + BATCH_SIZE]
    rows = []
    for _, row in batch.iterrows():
        vals = []
        for col in flat_columns:
            v = row[col]
            if pd.isna(v):
                vals.append(None)
            elif col in ("order_date", "ship_date"):
                vals.append(v.date() if hasattr(v, "date") else v)
            elif col in ("sales", "profit", "profit_margin", "discount"):
                vals.append(float(v))
            elif col in ("row_id", "shipping_days", "quantity",
                         "order_year", "order_month", "order_quarter", "is_loss"):
                vals.append(int(v))
            else:
                vals.append(str(v))
        rows.append(tuple(vals))

    extras.execute_batch(cur, insert_sql, rows)
    conn.commit()
    total_inserted += len(rows)
    logger.debug(f"  Batch inserted rows {batch_start+1}-{batch_start+len(rows)}")

elapsed = time.time() - start_time
logger.info(f"Inserted {total_inserted:,} rows in {elapsed:.2f}s "
            f"({total_inserted/elapsed:,.0f} rows/sec)")


# Step 6: Create the star schema (dimensions + fact table)
section("6. Creating Star Schema")

star_ddl = """
CREATE TABLE dim_customer (
    customer_sk     SERIAL          PRIMARY KEY,
    customer_id     VARCHAR(20)     NOT NULL UNIQUE,
    customer_name   VARCHAR(100)    NOT NULL,
    segment         VARCHAR(20)     NOT NULL
);

CREATE TABLE dim_product (
    product_sk      SERIAL          PRIMARY KEY,
    product_id      VARCHAR(30)     NOT NULL UNIQUE,
    product_name    TEXT            NOT NULL,
    category        VARCHAR(30)     NOT NULL,
    sub_category    VARCHAR(30)     NOT NULL
);

CREATE TABLE dim_geography (
    geography_sk    SERIAL          PRIMARY KEY,
    postal_code     VARCHAR(10)     NOT NULL UNIQUE,
    city            VARCHAR(100)    NOT NULL,
    state           VARCHAR(50)     NOT NULL,
    country         VARCHAR(50)     NOT NULL DEFAULT 'United States',
    region          VARCHAR(20)     NOT NULL
);

CREATE TABLE dim_date (
    date_sk         SERIAL          PRIMARY KEY,
    full_date       DATE            NOT NULL UNIQUE,
    year            SMALLINT        NOT NULL,
    quarter         SMALLINT        NOT NULL,
    month           SMALLINT        NOT NULL,
    month_name      VARCHAR(10)     NOT NULL,
    day_of_month    SMALLINT        NOT NULL,
    day_of_week     SMALLINT        NOT NULL,
    day_name        VARCHAR(10)     NOT NULL,
    is_weekend      BOOLEAN         NOT NULL
);

CREATE TABLE dim_ship_mode (
    ship_mode_sk    SERIAL          PRIMARY KEY,
    ship_mode       VARCHAR(20)     NOT NULL UNIQUE
);

CREATE TABLE fact_sales (
    sale_id         INTEGER         PRIMARY KEY,
    order_id        VARCHAR(20)     NOT NULL,
    customer_sk     INTEGER         NOT NULL REFERENCES dim_customer(customer_sk),
    product_sk      INTEGER         NOT NULL REFERENCES dim_product(product_sk),
    geography_sk    INTEGER         NOT NULL REFERENCES dim_geography(geography_sk),
    order_date_sk   INTEGER         NOT NULL REFERENCES dim_date(date_sk),
    ship_date_sk    INTEGER         NOT NULL REFERENCES dim_date(date_sk),
    ship_mode_sk    INTEGER         NOT NULL REFERENCES dim_ship_mode(ship_mode_sk),
    sales           NUMERIC(12,4)   NOT NULL,
    quantity         SMALLINT        NOT NULL,
    discount        NUMERIC(4,2)    NOT NULL,
    profit          NUMERIC(12,4)   NOT NULL,
    shipping_days   SMALLINT        NOT NULL,
    profit_margin   NUMERIC(8,4)    NOT NULL,
    is_loss         SMALLINT        NOT NULL,
    discount_tier   VARCHAR(30)     NOT NULL
);

CREATE INDEX idx_fact_customer   ON fact_sales (customer_sk);
CREATE INDEX idx_fact_product    ON fact_sales (product_sk);
CREATE INDEX idx_fact_geography  ON fact_sales (geography_sk);
CREATE INDEX idx_fact_order_date ON fact_sales (order_date_sk);
CREATE INDEX idx_fact_ship_date  ON fact_sales (ship_date_sk);
CREATE INDEX idx_fact_ship_mode  ON fact_sales (ship_mode_sk);
CREATE INDEX idx_fact_order_id   ON fact_sales (order_id);
CREATE INDEX idx_fact_profit     ON fact_sales (profit);
CREATE INDEX idx_fact_is_loss    ON fact_sales (is_loss);
"""

cur.execute(star_ddl)
conn.commit()
logger.info("Star schema created: 5 dimension tables + 1 fact table")


# Step 7: Populate dimension tables from the dataframe
section("7. Populating Dimension Tables")

# dim_customer - one row per unique customer
customers = df_db[["customer_id", "customer_name", "segment"]].drop_duplicates(
    subset=["customer_id"], keep="first").sort_values("customer_id")
cust_rows = [tuple(r) for r in customers.values]
extras.execute_batch(cur,
    "INSERT INTO dim_customer (customer_id, customer_name, segment) VALUES (%s, %s, %s)",
    cust_rows)
conn.commit()
logger.info(f"dim_customer: {len(cust_rows):,} rows inserted")

# dim_product - one row per unique product
products = df_db[["product_id", "product_name", "category", "sub_category"]].drop_duplicates(
    subset=["product_id"], keep="first").sort_values("product_id")
prod_rows = [tuple(r) for r in products.values]
extras.execute_batch(cur,
    "INSERT INTO dim_product (product_id, product_name, category, sub_category) VALUES (%s, %s, %s, %s)",
    prod_rows)
conn.commit()
logger.info(f"dim_product: {len(prod_rows):,} rows inserted")

# dim_geography - one row per postal code
geo = df_db[["postal_code", "city", "state", "country", "region"]].drop_duplicates(
    subset=["postal_code"], keep="first").sort_values("postal_code")
geo_rows = [tuple(r) for r in geo.values]
extras.execute_batch(cur,
    "INSERT INTO dim_geography (postal_code, city, state, country, region) VALUES (%s, %s, %s, %s, %s)",
    geo_rows)
conn.commit()
logger.info(f"dim_geography: {len(geo_rows):,} rows inserted")

# dim_date - one row per unique date (from both order and ship dates)
all_dates = pd.concat([df["Order Date"], df["Ship Date"]]).drop_duplicates().sort_values()
date_rows = []
for d in all_dates:
    date_rows.append((
        d.date(), d.year, (d.month - 1) // 3 + 1, d.month,
        d.strftime("%B"), d.day, d.weekday() + 1, d.strftime("%A"),
        d.weekday() >= 5
    ))
extras.execute_batch(cur,
    """INSERT INTO dim_date (full_date, year, quarter, month, month_name,
       day_of_month, day_of_week, day_name, is_weekend)
       VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)""",
    date_rows)
conn.commit()
logger.info(f"dim_date: {len(date_rows):,} rows inserted")

# dim_ship_mode - the 4 shipping methods
ship_modes = sorted(df["Ship Mode"].unique())
for mode in ship_modes:
    cur.execute("INSERT INTO dim_ship_mode (ship_mode) VALUES (%s)", (mode,))
conn.commit()
logger.info(f"dim_ship_mode: {len(ship_modes)} rows inserted")


# Step 8: Populate the fact table using surrogate key lookups
section("8. Populating Fact Table (fact_sales)")

# Build lookup dicts for surrogate keys
cur.execute("SELECT customer_id, customer_sk FROM dim_customer")
cust_sk = dict(cur.fetchall())

cur.execute("SELECT product_id, product_sk FROM dim_product")
prod_sk = dict(cur.fetchall())

cur.execute("SELECT postal_code, geography_sk FROM dim_geography")
geo_sk = dict(cur.fetchall())

cur.execute("SELECT full_date, date_sk FROM dim_date")
date_sk = {str(k): v for k, v in cur.fetchall()}

cur.execute("SELECT ship_mode, ship_mode_sk FROM dim_ship_mode")
mode_sk = dict(cur.fetchall())

# Build and insert fact rows
fact_rows = []
for _, row in df_db.iterrows():
    fact_rows.append((
        int(row["row_id"]),
        str(row["order_id"]),
        cust_sk[str(row["customer_id"])],
        prod_sk[str(row["product_id"])],
        geo_sk[str(row["postal_code"])],
        date_sk[str(row["order_date"].date())],
        date_sk[str(row["ship_date"].date())],
        mode_sk[str(row["ship_mode"])],
        float(row["sales"]),
        int(row["quantity"]),
        float(row["discount"]),
        float(row["profit"]),
        int(row["shipping_days"]),
        float(row["profit_margin"]),
        int(row["is_loss"]),
        str(row["discount_tier"])
    ))

start_time = time.time()
for batch_start in range(0, len(fact_rows), BATCH_SIZE):
    batch = fact_rows[batch_start:batch_start + BATCH_SIZE]
    extras.execute_batch(cur,
        """INSERT INTO fact_sales
           (sale_id, order_id, customer_sk, product_sk, geography_sk,
            order_date_sk, ship_date_sk, ship_mode_sk,
            sales, quantity, discount, profit, shipping_days,
            profit_margin, is_loss, discount_tier)
           VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
        batch)
    conn.commit()

elapsed = time.time() - start_time
logger.info(f"fact_sales: {len(fact_rows):,} rows inserted in {elapsed:.2f}s "
            f"({len(fact_rows)/elapsed:,.0f} rows/sec)")


# Step 9: Create analytical views for common queries
section("9. Creating Analytical Views")

views_sql = """
CREATE OR REPLACE VIEW v_monthly_sales AS
SELECT
    order_year, order_month,
    COUNT(*)              AS transactions,
    SUM(sales)            AS total_sales,
    SUM(profit)           AS total_profit,
    AVG(profit_margin)    AS avg_margin,
    SUM(is_loss)          AS loss_transactions
FROM sales_data
GROUP BY order_year, order_month
ORDER BY order_year, order_month;

CREATE OR REPLACE VIEW v_regional_performance AS
SELECT
    region, state,
    COUNT(*)              AS transactions,
    SUM(sales)            AS total_sales,
    SUM(profit)           AS total_profit,
    ROUND(AVG(profit_margin), 2) AS avg_margin_pct,
    ROUND(SUM(is_loss)::NUMERIC / COUNT(*) * 100, 2) AS loss_rate_pct
FROM sales_data
GROUP BY region, state
ORDER BY total_sales DESC;

CREATE OR REPLACE VIEW v_category_performance AS
SELECT
    category, sub_category,
    COUNT(*)              AS transactions,
    SUM(sales)            AS total_sales,
    SUM(profit)           AS total_profit,
    ROUND(AVG(profit_margin), 2) AS avg_margin_pct,
    SUM(is_loss)          AS loss_count
FROM sales_data
GROUP BY category, sub_category
ORDER BY total_sales DESC;

CREATE OR REPLACE VIEW v_top_customers AS
SELECT
    customer_id, customer_name, segment,
    COUNT(*)              AS order_count,
    SUM(sales)            AS total_spent,
    SUM(profit)           AS total_profit_generated,
    ROUND(AVG(sales), 2)  AS avg_order_value
FROM sales_data
GROUP BY customer_id, customer_name, segment
ORDER BY total_spent DESC
LIMIT 100;
"""

cur.execute(views_sql)
conn.commit()
logger.info("Created 4 analytical views: v_monthly_sales, v_regional_performance, "
            "v_category_performance, v_top_customers")


# Step 10: Run verification checks
section("10. Data Import Verification")

checks_passed = 0
checks_total  = 0

def verify(description, query, expected=None):
    global checks_passed, checks_total
    checks_total += 1
    cur.execute(query)
    result = cur.fetchone()[0]
    status = "PASS" if (expected is None or str(result) == str(expected)) else "FAIL"
    if status == "PASS":
        checks_passed += 1
    logger.info(f"  [{status}] {description}: {result:,}"
                f"{'  (expected: ' + str(expected) + ')' if expected is not None and status == 'FAIL' else ''}")
    return result

# Row counts
logger.info("-- Row Count Verification --")
verify("sales_data row count",
       "SELECT COUNT(*) FROM sales_data", len(df))
verify("fact_sales row count",
       "SELECT COUNT(*) FROM fact_sales", len(df))
verify("dim_customer row count",
       "SELECT COUNT(*) FROM dim_customer", len(customers))
verify("dim_product row count",
       "SELECT COUNT(*) FROM dim_product", len(products))
verify("dim_geography row count",
       "SELECT COUNT(*) FROM dim_geography", len(geo))
verify("dim_date row count",
       "SELECT COUNT(*) FROM dim_date", len(date_rows))
verify("dim_ship_mode row count",
       "SELECT COUNT(*) FROM dim_ship_mode", len(ship_modes))

# Aggregate totals match CSV
logger.info("-- Aggregate Consistency (Flat vs CSV) --")
csv_total_sales  = round(df["Sales"].sum(), 2)
csv_total_profit = round(df["Profit"].sum(), 2)

verify("Total Sales (sales_data)",
       "SELECT ROUND(SUM(sales)::NUMERIC, 2) FROM sales_data", csv_total_sales)
verify("Total Profit (sales_data)",
       "SELECT ROUND(SUM(profit)::NUMERIC, 2) FROM sales_data", csv_total_profit)

# Flat table matches star schema
logger.info("-- Cross-Schema Consistency (Flat vs Star) --")
verify("Flat total sales matches star total",
       """SELECT CASE WHEN
           (SELECT SUM(sales) FROM sales_data) =
           (SELECT SUM(sales) FROM fact_sales)
           THEN 1 ELSE 0 END""", 1)
verify("Flat total profit matches star total",
       """SELECT CASE WHEN
           (SELECT SUM(profit) FROM sales_data) =
           (SELECT SUM(profit) FROM fact_sales)
           THEN 1 ELSE 0 END""", 1)

# Constraint and referential integrity
logger.info("-- Constraint & Integrity Checks --")
verify("No negative sales values",
       "SELECT COUNT(*) FROM sales_data WHERE sales < 0", 0)
verify("No negative quantities",
       "SELECT COUNT(*) FROM sales_data WHERE quantity <= 0", 0)
verify("No invalid discounts (>1)",
       "SELECT COUNT(*) FROM sales_data WHERE discount > 1", 0)
verify("No ship before order",
       "SELECT COUNT(*) FROM sales_data WHERE ship_date < order_date", 0)
verify("Orphaned fact rows (customer)",
       """SELECT COUNT(*) FROM fact_sales f
          LEFT JOIN dim_customer c ON f.customer_sk = c.customer_sk
          WHERE c.customer_sk IS NULL""", 0)
verify("Orphaned fact rows (product)",
       """SELECT COUNT(*) FROM fact_sales f
          LEFT JOIN dim_product p ON f.product_sk = p.product_sk
          WHERE p.product_sk IS NULL""", 0)

# Sample data preview
logger.info("-- Sample Data Preview (first 3 rows) --")
cur.execute("""SELECT row_id, order_id, order_date, customer_name,
               category, sales, profit FROM sales_data ORDER BY row_id LIMIT 3""")
for row in cur.fetchall():
    logger.info(f"  Row {row[0]}: {row[1]} | {row[2]} | {row[3]} | {row[4]} | "
                f"${row[5]:,.2f} | ${row[6]:,.2f}")

# Table sizes
logger.info("-- Table Sizes --")
tables = ["sales_data", "fact_sales", "dim_customer", "dim_product",
          "dim_geography", "dim_date", "dim_ship_mode"]
for tbl in tables:
    cur.execute(f"SELECT pg_size_pretty(pg_total_relation_size('{tbl}'))")
    size = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    cnt = cur.fetchone()[0]
    logger.info(f"  {tbl:20s} -> {cnt:>6,} rows  |  {size}")


# Step 11: Print final summary
section("11. Import Summary")

logger.info(f"Database         : {DB_NAME}")
logger.info(f"Host             : {DB_HOST}:{DB_PORT}")
logger.info(f"User             : {DB_USER}")
logger.info(f"CSV Source       : {CSV_FILE} ({len(df):,} rows)")
logger.info(f"Flat Table       : sales_data ({len(df):,} rows, 29 cols)")
logger.info(f"Star Schema      : 5 dimensions + 1 fact table ({len(fact_rows):,} facts)")
logger.info(f"Views Created    : 4 analytical views")
logger.info(f"Verification     : {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    logger.info("ALL VERIFICATION CHECKS PASSED - Database is ready!")
else:
    logger.warning(f"{checks_total - checks_passed} check(s) FAILED - review log")

cur.close()
conn.close()
logger.info("Connection closed.")


## **Pull Metrics**

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.abspath(os.path.join(os.path.abspath('.'), '..')), '.env'))

import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname=os.environ.get("DB_NAME", "working_database"),
    user=os.environ.get("DB_USER", "postgres"),
    password=os.environ.get("DB_PASSWORD", "your_password_here"),
    host=os.environ.get("DB_HOST", "localhost"),
    port=os.environ.get("DB_PORT", "5432")
)

queries = {
    "category_perf": """
        SELECT category, SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin,
               COUNT(*) AS txns, SUM(is_loss) AS losses
        FROM sales_data GROUP BY category ORDER BY sales DESC""",

    "subcat_profit": """
        SELECT sub_category, SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin
        FROM sales_data GROUP BY sub_category ORDER BY profit""",

    "region_perf": """
        SELECT region, SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin,
               COUNT(DISTINCT customer_id) AS customers,
               SUM(is_loss) AS losses, COUNT(*) AS txns
        FROM sales_data GROUP BY region ORDER BY sales DESC""",

    "state_perf": """
        SELECT state, region, SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin
        FROM sales_data GROUP BY state, region ORDER BY profit LIMIT 10""",

    "worst_states": """
        SELECT state, region, SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin
        FROM sales_data GROUP BY state, region ORDER BY profit ASC LIMIT 10""",

    "yoy_growth": """
        SELECT order_year, SUM(sales) AS sales, SUM(profit) AS profit,
               COUNT(DISTINCT order_id) AS orders
        FROM sales_data GROUP BY order_year ORDER BY order_year""",

    "discount_impact": """
        SELECT discount_tier, COUNT(*) AS txns, SUM(sales) AS sales,
               SUM(profit) AS profit, SUM(is_loss) AS losses,
               ROUND((SUM(is_loss)::numeric/COUNT(*)*100), 1) AS loss_rate
        FROM sales_data GROUP BY discount_tier ORDER BY MIN(discount)""",

    "customer_retention": """
        SELECT CASE WHEN order_count = 1 THEN 'One-time'
                    WHEN order_count BETWEEN 2 AND 5 THEN 'Repeat (2-5)'
                    WHEN order_count BETWEEN 6 AND 10 THEN 'Loyal (6-10)'
                    ELSE 'VIP (11+)' END AS tier,
               COUNT(*) AS customers, SUM(total_spent) AS revenue
        FROM (SELECT customer_id, COUNT(DISTINCT order_id) AS order_count,
                     SUM(sales) AS total_spent FROM sales_data
              GROUP BY customer_id) t
        GROUP BY 1 ORDER BY MIN(order_count)""",

    "top_loss_products": """
        SELECT product_name, sub_category, SUM(profit) AS total_loss,
               COUNT(*) AS times, ROUND(AVG(discount)::numeric, 2) AS avg_disc
        FROM sales_data WHERE is_loss = 1
        GROUP BY product_name, sub_category
        HAVING COUNT(*) >= 3 ORDER BY total_loss LIMIT 10""",

    "segment_perf": """
        SELECT segment, COUNT(DISTINCT customer_id) AS customers,
               SUM(sales) AS sales, SUM(profit) AS profit,
               ROUND((SUM(profit)/SUM(sales)*100)::numeric, 1) AS margin
        FROM sales_data GROUP BY segment ORDER BY sales DESC""",

    "monthly_trend": """
        SELECT order_year, order_month, SUM(sales) AS sales, SUM(profit) AS profit
        FROM sales_data GROUP BY order_year, order_month ORDER BY order_year, order_month""",

    "ship_mode_perf": """
        SELECT ship_mode, COUNT(*) AS txns, SUM(sales) AS sales,
               SUM(profit) AS profit, AVG(shipping_days) AS avg_days
        FROM sales_data GROUP BY ship_mode ORDER BY sales DESC""",

    "q4_vs_other": """
        SELECT CASE WHEN order_quarter = 4 THEN 'Q4' ELSE 'Q1-Q3' END AS period,
               SUM(sales) AS sales, SUM(profit) AS profit, COUNT(DISTINCT order_id) AS orders
        FROM sales_data GROUP BY 1 ORDER BY period"""
}

for name, sql in queries.items():
    df = pd.read_sql(sql, conn)
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(df.to_string(index=False))

conn.close()


## **Advanced Analytics**

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, shapiro
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

# Configuration
PROJECT_ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_data.csv")
VIZ_DIR = os.path.join(PROJECT_ROOT, "visualizations", "advanced")
os.makedirs(VIZ_DIR, exist_ok=True)

# Style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight", "figure.facecolor": "white"})
PAL = ["#2563eb", "#f59e0b", "#10b981", "#ef4444", "#8b5cf6", "#06b6d4", "#ec4899", "#84cc16"]


def save(fig, name):
    fig.savefig(os.path.join(VIZ_DIR, f"{name}.png"), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {name}.png")


# Load data
print("Loading data...")
df = pd.read_csv(DATA_PATH, encoding="utf-8")
# Standardize column names to snake_case
df.columns = [c.strip().replace(" ", "_").replace("-", "_").lower() for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df["order_yearmonth"] = df["order_date"].dt.to_period("M")
print(f"Loaded {len(df):,} rows\n")



# SECTION 1: TREND ANALYSIS

print("=" * 60)
print("SECTION 1: TREND ANALYSIS")
print("=" * 60)

# 1.1 Monthly time series preparation
monthly = df.groupby("order_yearmonth").agg(
    revenue=("sales", "sum"),
    profit=("profit", "sum"),
    orders=("order_id", "nunique"),
    transactions=("sales", "count")
).reset_index()
monthly["date"] = monthly["order_yearmonth"].dt.to_timestamp()
monthly = monthly.sort_values("date").reset_index(drop=True)

# 1.2 Time Series Decomposition
print("  1.2 Time Series Decomposition")
decomp = seasonal_decompose(monthly["revenue"], model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Time Series Decomposition — Monthly Revenue", fontsize=15, fontweight="bold")
axes[0].plot(monthly["date"], decomp.observed, color=PAL[0], linewidth=1.5)
axes[0].set_ylabel("Observed")
axes[1].plot(monthly["date"], decomp.trend, color=PAL[3], linewidth=2)
axes[1].set_ylabel("Trend")
axes[2].plot(monthly["date"], decomp.seasonal, color=PAL[2], linewidth=1)
axes[2].set_ylabel("Seasonal")
axes[3].plot(monthly["date"], decomp.resid, color=PAL[4], linewidth=0.8, alpha=0.7)
axes[3].axhline(0, color="black", linewidth=0.5)
axes[3].set_ylabel("Residual")
plt.tight_layout()
save(fig, "01_time_series_decomposition")

# 1.3 Moving Averages + Momentum
print("  1.3 Moving Averages & Momentum")
monthly["MA3"] = monthly["revenue"].rolling(3).mean()
monthly["MA6"] = monthly["revenue"].rolling(6).mean()
monthly["MA12"] = monthly["revenue"].rolling(12).mean()
monthly["momentum"] = monthly["revenue"].diff()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))
ax1.plot(monthly["date"], monthly["revenue"], color=PAL[0], alpha=0.4, linewidth=1, label="Actual")
ax1.plot(monthly["date"], monthly["MA3"], color=PAL[3], linewidth=2, label="3-Month MA")
ax1.plot(monthly["date"], monthly["MA6"], color=PAL[1], linewidth=2, label="6-Month MA")
ax1.plot(monthly["date"], monthly["MA12"], color=PAL[4], linewidth=2.5, label="12-Month MA")
ax1.set_title("Revenue with Moving Averages", fontsize=14, fontweight="bold")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax1.legend()

colors = [PAL[2] if m >= 0 else PAL[3] for m in monthly["momentum"].fillna(0)]
ax2.bar(monthly["date"], monthly["momentum"], color=colors, alpha=0.8, width=25)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Revenue Momentum (Month-over-Month Change)", fontsize=14, fontweight="bold")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
plt.tight_layout()
save(fig, "02_moving_averages_momentum")

# 1.4 YoY and MoM Growth with Confidence Intervals
print("  1.4 Growth Rates with Confidence Intervals")
monthly["yoy_growth"] = monthly["revenue"].pct_change(12) * 100
monthly["mom_growth"] = monthly["revenue"].pct_change(1) * 100
yoy_mean = monthly["yoy_growth"].dropna().mean()
yoy_ci = stats.t.interval(0.95, len(monthly["yoy_growth"].dropna()) - 1,
                           loc=yoy_mean, scale=stats.sem(monthly["yoy_growth"].dropna()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(monthly["date"].iloc[12:], monthly["yoy_growth"].iloc[12:],
        color=[PAL[2] if g >= 0 else PAL[3] for g in monthly["yoy_growth"].iloc[12:]], alpha=0.85, width=25)
ax1.axhline(yoy_mean, color=PAL[4], linewidth=2, linestyle="--", label=f"Mean: {yoy_mean:.1f}%")
ax1.axhspan(yoy_ci[0], yoy_ci[1], alpha=0.15, color=PAL[4], label=f"95% CI: [{yoy_ci[0]:.1f}%, {yoy_ci[1]:.1f}%]")
ax1.set_title("Year-over-Year Revenue Growth", fontsize=13, fontweight="bold")
ax1.set_ylabel("Growth %")
ax1.legend(fontsize=9)

ax2.bar(monthly["date"].iloc[1:], monthly["mom_growth"].iloc[1:],
        color=[PAL[2] if g >= 0 else PAL[3] for g in monthly["mom_growth"].iloc[1:]], alpha=0.85, width=25)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Month-over-Month Revenue Growth", fontsize=13, fontweight="bold")
ax2.set_ylabel("Growth %")
plt.tight_layout()
save(fig, "03_growth_rates")

# 1.5 Exponential Smoothing Forecast
print("  1.5 Sales Forecasting (Holt-Winters)")
monthly_ts = monthly.set_index("date")["revenue"].asfreq("MS")
monthly_ts = monthly_ts.interpolate()

model = ExponentialSmoothing(monthly_ts, trend="add", seasonal="add", seasonal_periods=12)
fit = model.fit(optimized=True)
forecast = fit.forecast(12)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_ts.index, monthly_ts.values, color=PAL[0], linewidth=1.5, label="Historical")
ax.plot(forecast.index, forecast.values, color=PAL[3], linewidth=2.5, linestyle="--", label="12-Month Forecast")
ax.fill_between(forecast.index, forecast.values * 0.85, forecast.values * 1.15,
                alpha=0.15, color=PAL[3], label="±15% Confidence Band")
ax.set_title("Revenue Forecast — Holt-Winters Exponential Smoothing", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.legend()
plt.tight_layout()
save(fig, "04_forecast_holt_winters")



# SECTION 2: CUSTOMER SEGMENTATION

print("\n" + "=" * 60)
print("SECTION 2: CUSTOMER SEGMENTATION")
print("=" * 60)

# 2.1 RFM Analysis
print("  2.1 RFM Analysis")
snapshot = df["order_date"].max() + pd.Timedelta(days=1)
rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (snapshot - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("sales", "sum")
).reset_index()

rfm["R_quartile"] = pd.qcut(rfm["recency"], 4, labels=["4-Best", "3", "2", "1-Worst"])
rfm["F_quartile"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=["1-Worst", "2", "3", "4-Best"])
rfm["M_quartile"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=["1-Worst", "2", "3", "4-Best"])


def rfm_segment(row):
    r, f, m = row["R_quartile"], row["F_quartile"], row["M_quartile"]
    if r in ["4-Best", "3"] and f in ["4-Best", "3"] and m in ["4-Best", "3"]:
        return "Champions"
    elif r in ["4-Best", "3"] and f in ["1-Worst", "2"]:
        return "New Customers"
    elif r in ["1-Worst", "2"] and f in ["4-Best", "3"]:
        return "At Risk"
    elif r in ["1-Worst", "2"] and f in ["1-Worst", "2"]:
        return "Lost"
    else:
        return "Potential Loyalists"


rfm["segment"] = rfm.apply(rfm_segment, axis=1)
seg_counts = rfm["segment"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
wedges, texts, autotexts = ax1.pie(seg_counts.values, labels=seg_counts.index,
                                    autopct="%1.1f%%", colors=PAL[:5], startangle=90, textprops={"fontsize": 10})
ax1.set_title("RFM Customer Segments", fontsize=14, fontweight="bold")

seg_revenue = rfm.groupby("segment")["monetary"].sum().sort_values(ascending=True)
bars = ax2.barh(seg_revenue.index, seg_revenue.values, color=PAL[:5], alpha=0.85)
ax2.set_title("Revenue by RFM Segment", fontsize=14, fontweight="bold")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
plt.tight_layout()
save(fig, "05_rfm_segments")

# 2.2 K-Means Clustering
print("  2.2 K-Means Clustering")
features = ["recency", "frequency", "monetary"]
scaler = StandardScaler()
X = scaler.fit_transform(rfm[features])

# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

# Fit with optimal k=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["cluster"] = km_final.fit_predict(X)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(list(K_range), inertias, color=PAL[0], marker="o", linewidth=2, markersize=8)
ax1.set_title("Elbow Method — Optimal K", fontsize=14, fontweight="bold")
ax1.set_xlabel("Number of Clusters")
ax1.set_ylabel("Inertia")
ax1.axvline(4, color=PAL[3], linestyle="--", linewidth=1.5, label="K=4 selected")
ax1.legend()

# Scatter: frequency vs monetary colored by cluster
scatter = ax2.scatter(rfm["frequency"], rfm["monetary"],
                      c=rfm["cluster"], cmap="viridis", alpha=0.6, s=30)
ax2.set_title("K-Means Clusters (Frequency vs Monetary)", fontsize=14, fontweight="bold")
ax2.set_xlabel("Frequency (orders)")
ax2.set_ylabel("Monetary ($)")
plt.colorbar(scatter, ax=ax2, label="Cluster")
plt.tight_layout()
save(fig, "06_kmeans_clustering")

# 2.3 Customer Lifetime Value
print("  2.3 Customer Lifetime Value")
clv = df.groupby("customer_id").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("sales", "sum"),
    avg_order_value=("sales", "mean"),
    first_order=("order_date", "min"),
    last_order=("order_date", "max")
).reset_index()
clv["lifespan_days"] = (clv["last_order"] - clv["first_order"]).dt.days
clv["purchase_rate"] = np.where(clv["lifespan_days"] > 0,
                                 clv["total_orders"] / clv["lifespan_days"] * 365,
                                 clv["total_orders"])
clv["estimated_clv"] = clv["avg_order_value"] * clv["purchase_rate"] * 3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(clv["estimated_clv"], bins=50, color=PAL[0], alpha=0.7, edgecolor="white")
ax1.axvline(clv["estimated_clv"].median(), color=PAL[3], linewidth=2, linestyle="--",
            label=f"Median CLV: ${clv['estimated_clv'].median():,.0f}")
ax1.axvline(clv["estimated_clv"].mean(), color=PAL[1], linewidth=2, linestyle="--",
            label=f"Mean CLV: ${clv['estimated_clv'].mean():,.0f}")
ax1.set_title("Customer Lifetime Value Distribution (3yr)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Estimated CLV ($)")
ax1.legend()

top_clv = clv.nlargest(20, "estimated_clv")
ax2.scatter(top_clv["total_orders"], top_clv["total_revenue"],
            s=top_clv["estimated_clv"] / 5, color=PAL[0], alpha=0.6, edgecolors="white")
ax2.set_title("Top 20 CLV Customers (size = CLV)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Total Orders")
ax2.set_ylabel("Total Revenue ($)")
plt.tight_layout()
save(fig, "07_customer_lifetime_value")

# 2.4 Cohort Retention Analysis
print("  2.4 Cohort Retention Analysis")
df["cohort"] = df.groupby("customer_id")["order_date"].transform("min").dt.to_period("Q")
df["order_period"] = df["order_date"].dt.to_period("Q")
df["cohort_index"] = (df["order_period"] - df["cohort"]).apply(lambda x: x.n)

cohort_data = df.groupby(["cohort", "cohort_index"])["customer_id"].nunique().reset_index()
cohort_data.rename(columns={"customer_id": "customers"}, inplace=True)
cohort_sizes = cohort_data.groupby("cohort")["customers"].max().reset_index()
cohort_sizes.rename(columns={"customers": "cohort_size"}, inplace=True)
cohort_data = cohort_data.merge(cohort_sizes)
cohort_data["retention_pct"] = (cohort_data["customers"] / cohort_data["cohort_size"] * 100).round(1)

pivot = cohort_data.pivot_table(index="cohort", columns="cohort_index",
                                 values="retention_pct", aggfunc="first")
pivot.columns = [f"Q{c}" for c in pivot.columns]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot.iloc[:, :8], annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Retention %"})
ax.set_title("Customer Cohort Retention Heatmap (%)", fontsize=15, fontweight="bold")
ax.set_xlabel("Quarters Since First Order")
ax.set_ylabel("Cohort (Quarter of First Order)")
plt.tight_layout()
save(fig, "08_cohort_retention")



# SECTION 3: PRODUCT ANALYSIS

print("\n" + "=" * 60)
print("SECTION 3: PRODUCT ANALYSIS")
print("=" * 60)

# 3.1 Product Performance Matrix (BCG-style)
print("  3.1 Product Performance Matrix")
prod = df.groupby(["product_id", "product_name", "category"]).agg(
    total_sales=("sales", "sum"),
    total_profit=("profit", "sum"),
    units_sold=("quantity", "sum"),
    transactions=("sales", "count")
).reset_index()

med_sales = prod["total_sales"].median()
med_profit = prod["total_profit"].median()


def classify_quadrant(row):
    if row["total_sales"] >= med_sales and row["total_profit"] >= med_profit:
        return "Stars"
    elif row["total_sales"] >= med_sales and row["total_profit"] < med_profit:
        return "Question Marks"
    elif row["total_sales"] < med_sales and row["total_profit"] >= med_profit:
        return "Cash Cows"
    else:
        return "Dogs"


prod["quadrant"] = prod.apply(classify_quadrant, axis=1)
quad_counts = prod["quadrant"].value_counts()

fig, ax = plt.subplots(figsize=(12, 9))
quad_colors = {"Stars": PAL[2], "Cash Cows": PAL[0], "Question Marks": PAL[1], "Dogs": PAL[3]}
for quad in ["Stars", "Cash Cows", "Question Marks", "Dogs"]:
    subset = prod[prod["quadrant"] == quad]
    ax.scatter(subset["total_sales"], subset["total_profit"],
               s=subset["units_sold"] * 2, alpha=0.5, color=quad_colors[quad],
               label=f"{quad} ({len(subset)})", edgecolors="white", linewidth=0.5)

ax.axhline(med_profit, color="gray", linewidth=1, linestyle="--", alpha=0.5)
ax.axvline(med_sales, color="gray", linewidth=1, linestyle="--", alpha=0.5)
ax.set_title("Product Performance Matrix (BCG-Style)", fontsize=15, fontweight="bold")
ax.set_xlabel("Total Sales ($)")
ax.set_ylabel("Total Profit ($)")
ax.legend(fontsize=11, loc="upper left")
plt.tight_layout()
save(fig, "09_product_matrix")

# 3.2 ABC Analysis
print("  3.2 ABC Analysis")
prod_sorted = prod.sort_values("total_sales", ascending=False).reset_index(drop=True)
prod_sorted["cumulative_pct"] = prod_sorted["total_sales"].cumsum() / prod_sorted["total_sales"].sum() * 100


def abc_class(pct):
    if pct <= 80:
        return "A"
    elif pct <= 95:
        return "B"
    else:
        return "C"


prod_sorted["abc"] = prod_sorted["cumulative_pct"].apply(abc_class)
abc_counts = prod_sorted["abc"].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
abc_colors = {"A": PAL[2], "B": PAL[1], "C": PAL[3]}
ax1.bar(abc_counts.index, abc_counts.values, color=[abc_colors[c] for c in abc_counts.index], alpha=0.85)
ax1.set_title("ABC Analysis — Product Count", fontsize=14, fontweight="bold")
ax1.set_xlabel("Class")
ax1.set_ylabel("Number of Products")
for i, v in enumerate(abc_counts.values):
    ax1.text(i, v + 5, str(v), ha="center", fontweight="bold")

ax2.plot(prod_sorted.index, prod_sorted["cumulative_pct"], color=PAL[0], linewidth=2)
ax2.axhline(80, color=PAL[2], linestyle="--", linewidth=1.5, label="80% threshold (A)")
ax2.axhline(95, color=PAL[1], linestyle="--", linewidth=1.5, label="95% threshold (B)")
ax2.fill_between(prod_sorted.index, 0, 80, alpha=0.1, color=PAL[2], label="Class A")
ax2.fill_between(prod_sorted.index, 80, 95, alpha=0.1, color=PAL[1], label="Class B")
ax2.fill_between(prod_sorted.index, 95, 100, alpha=0.1, color=PAL[3], label="Class C")
ax2.set_title("ABC Cumulative Revenue Curve", fontsize=14, fontweight="bold")
ax2.set_xlabel("Products (sorted by revenue)")
ax2.set_ylabel("Cumulative Revenue %")
ax2.legend(fontsize=9)
plt.tight_layout()
save(fig, "10_abc_analysis")

# 3.3 Price Elasticity (Discount vs Sales Volume)
print("  3.3 Price Elasticity Analysis")
elasticity = df.groupby("discount").agg(
    avg_sales=("sales", "mean"),
    avg_quantity=("quantity", "mean"),
    avg_profit=("profit", "mean"),
    transactions=("sales", "count")
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(elasticity["discount"] * 100, elasticity["avg_sales"],
            s=elasticity["transactions"] / 5, color=PAL[0], alpha=0.7, edgecolors="white")
z = np.polyfit(elasticity["discount"] * 100, elasticity["avg_sales"], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 80, 100)
ax1.plot(x_line, p(x_line), color=PAL[3], linewidth=2, linestyle="--",
         label=f"Trend (slope={z[0]:.1f})")
ax1.set_title("Discount % vs Avg Sale Value", fontsize=13, fontweight="bold")
ax1.set_xlabel("Discount (%)")
ax1.set_ylabel("Avg Sale ($)")
ax1.legend()

ax2.scatter(elasticity["discount"] * 100, elasticity["avg_profit"],
            s=elasticity["transactions"] / 5, color=[PAL[2] if p >= 0 else PAL[3] for p in elasticity["avg_profit"]],
            alpha=0.7, edgecolors="white")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Discount % vs Avg Profit", fontsize=13, fontweight="bold")
ax2.set_xlabel("Discount (%)")
ax2.set_ylabel("Avg Profit ($)")
plt.tight_layout()
save(fig, "11_price_elasticity")

# 3.4 Cross-Selling Opportunities (products frequently in same order)
print("  3.4 Cross-Selling Opportunities")
order_products = df.groupby("order_id")["sub_category"].apply(list).reset_index()
from collections import Counter
from itertools import combinations

pair_counter = Counter()
for _, row in order_products.iterrows():
    items = sorted(set(row["sub_category"]))
    for pair in combinations(items, 2):
        pair_counter[pair] += 1

top_pairs = pair_counter.most_common(15)
pairs_df = pd.DataFrame(top_pairs, columns=["pair", "count"])
pairs_df["label"] = pairs_df["pair"].apply(lambda x: f"{x[0]} + {x[1]}")

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(pairs_df)-1, -1, -1), pairs_df["count"], color=PAL[5], alpha=0.85)
ax.set_yticks(range(len(pairs_df)-1, -1, -1))
ax.set_yticklabels(pairs_df["label"], fontsize=10)
ax.set_title("Top 15 Cross-Selling Opportunities (Product Pairs in Same Order)", fontsize=14, fontweight="bold")
ax.set_xlabel("Co-occurrence Count")
plt.tight_layout()
save(fig, "12_cross_selling")



# SECTION 4: ADVANCED STATISTICAL ANALYSIS

print("\n" + "=" * 60)
print("SECTION 4: ADVANCED STATISTICAL ANALYSIS")
print("=" * 60)

# 4.1 Correlation Matrix
print("  4.1 Correlation Matrix")
numeric_cols = ["sales", "quantity", "discount", "profit", "shipping_days",
                "profit_margin", "order_year", "order_month"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            square=True, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix — Numerical Variables", fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "13_correlation_matrix")

# 4.2 Hypothesis Testing
print("  4.2 Hypothesis Testing")
results = []

# Test 1: Do discounted orders generate more revenue?
discounted = df[df["discount"] > 0]["sales"]
no_discount = df[df["discount"] == 0]["sales"]
t_stat, p_val = ttest_ind(discounted, no_discount, equal_var=False)
results.append(("Discounted vs No-Discount Sales", f"t={t_stat:.2f}", f"p={p_val:.4f}",
                "Significant" if p_val < 0.05 else "Not Significant"))

# Test 2: Profit difference between regions
regions = df["region"].unique()
west = df[df["region"] == "West"]["profit"]
east = df[df["region"] == "East"]["profit"]
t_stat2, p_val2 = ttest_ind(west, east, equal_var=False)
results.append(("West vs East Profit", f"t={t_stat2:.2f}", f"p={p_val2:.4f}",
                "Significant" if p_val2 < 0.05 else "Not Significant"))

# Test 3: Profit by segment
consumer = df[df["segment"] == "Consumer"]["profit"]
corporate = df[df["segment"] == "Corporate"]["profit"]
t_stat3, p_val3 = ttest_ind(consumer, corporate, equal_var=False)
results.append(("Consumer vs Corporate Profit", f"t={t_stat3:.2f}", f"p={p_val3:.4f}",
                "Significant" if p_val3 < 0.05 else "Not Significant"))

# Test 4: Do higher quantities yield higher profit margins?
high_qty = df[df["quantity"] >= 5]["profit_margin"]
low_qty = df[df["quantity"] < 5]["profit_margin"]
t_stat4, p_val4 = ttest_ind(high_qty, low_qty, equal_var=False)
results.append(("High Qty vs Low Qty Margin", f"t={t_stat4:.2f}", f"p={p_val4:.4f}",
                "Significant" if p_val4 < 0.05 else "Not Significant"))

ht_df = pd.DataFrame(results, columns=["Test", "Statistic", "P-Value", "Result (α=0.05)"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
table = ax.table(cellText=ht_df.values, colLabels=ht_df.columns, loc="center",
                 cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor(PAL[0])
        cell.set_text_props(color="white", fontweight="bold")
    elif col == 3:
        cell.set_facecolor("#d4edda" if cell.get_text().get_text() == "Significant" else "#f8d7da")
ax.set_title("Hypothesis Testing Results", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
save(fig, "14_hypothesis_testing")

# 4.3 Predictive Modeling (Gradient Boosting)
print("  4.3 Predictive Modeling — Sales Forecast")
model_df = df.copy()
model_df["day_of_year"] = model_df["order_date"].dt.dayofyear
model_df["week_of_year"] = model_df["order_date"].dt.isocalendar().week.astype(int)
model_df["day_of_week"] = model_df["order_date"].dt.dayofweek

features = ["quantity", "discount", "shipping_days", "order_year", "order_month",
            "day_of_year", "week_of_year", "day_of_week"]
X = model_df[features].values
y = model_df["sales"].values

tscv = TimeSeriesSplit(n_splits=5)
mae_scores, r2_scores = [], []

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
    gb.fit(X_train, y_train)
    y_pred = gb.predict(X_test)
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    r2_scores.append(r2_score(y_test, y_pred))

# Feature importance
gb_full = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
gb_full.fit(X, y)
importance = pd.Series(gb_full.feature_importances_, index=features).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.barh(importance.index, importance.values, color=PAL[0], alpha=0.85)
ax1.set_title("Feature Importance — Sales Prediction (GBR)", fontsize=13, fontweight="bold")

# Actual vs Predicted on last fold
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
y_pred = gb_full.predict(X_test)
sample = np.random.choice(len(y_test), min(500, len(y_test)), replace=False)
ax2.scatter(y_test[sample], y_pred[sample], alpha=0.3, s=15, color=PAL[0])
max_val = max(y_test.max(), y_pred.max())
ax2.plot([0, max_val], [0, max_val], color=PAL[3], linewidth=2, linestyle="--", label="Perfect prediction")
ax2.set_title(f"Actual vs Predicted Sales\nMAE={np.mean(mae_scores):.1f} | R²={np.mean(r2_scores):.2f}",
              fontsize=13, fontweight="bold")
ax2.set_xlabel("Actual Sales ($)")
ax2.set_ylabel("Predicted Sales ($)")
ax2.legend()
plt.tight_layout()
save(fig, "15_predictive_modeling")

# 4.4 Outlier Deep Dive
print("  4.4 Outlier Analysis")
from sklearn.ensemble import IsolationForest

iso_features = ["sales", "quantity", "discount", "profit"]
iso_scaler = RobustScaler()
X_iso = iso_scaler.fit_transform(df[iso_features])

iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
df["anomaly"] = iso.fit_predict(X_iso)
anomalies = df[df["anomaly"] == -1]
normal = df[df["anomaly"] == 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, iso_features):
    ax.hist(normal[feat], bins=50, alpha=0.5, color=PAL[0], label=f"Normal ({len(normal):,})")
    ax.hist(anomalies[feat], bins=50, alpha=0.7, color=PAL[3], label=f"Anomaly ({len(anomalies):,})")
    ax.set_title(feat, fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
fig.suptitle("Outlier Detection — Isolation Forest (3% contamination)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
save(fig, "16_outlier_analysis")



# SUMMARY

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print(f"Charts saved to: {VIZ_DIR}/")
print(f"Total charts generated: 16")
print(f"\nKey findings:")
print(f"  - YoY avg growth: {yoy_mean:.1f}% (95% CI: [{yoy_ci[0]:.1f}%, {yoy_ci[1]:.1f}%])")
print(f"  - RFM segments: {dict(seg_counts)}")
print(f"  - K-Means optimal clusters: 4")
print(f"  - Median CLV (3yr): ${clv['estimated_clv'].median():,.0f}")
print(f"  - ABC: {dict(abc_counts)} products")
print(f"  - Model MAE: {np.mean(mae_scores):.1f}, R²: {np.mean(r2_scores):.2f}")
print(f"  - Anomalies detected: {len(anomalies)} ({len(anomalies)/len(df)*100:.1f}%)")


## **Showcase Visualizations**

In [ ]:
import os
import sys
import warnings
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.abspath(os.path.join(os.path.abspath('.'), '..')), '.env'))

import numpy as np
import pandas as pd
import psycopg2
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

# Configuration
PROJECT_ROOT = os.path.abspath('..')
VIZ_DIR = os.path.join(PROJECT_ROOT, "visualizations")
os.makedirs(VIZ_DIR, exist_ok=True)

DB_PARAMS = dict(
    dbname=os.environ.get("DB_NAME", "working_database"),
    user=os.environ.get("DB_USER", "postgres"),
    password=os.environ.get("DB_PASSWORD", "your_password_here"),
    host=os.environ.get("DB_HOST", "localhost"),
    port=os.environ.get("DB_PORT", "5432")
)

# Chart style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["figure.facecolor"] = "white"

# Colors
PALETTE = ["#2563eb", "#f59e0b", "#10b981", "#ef4444", "#8b5cf6", "#06b6d4"]


def run_query(sql):
    """Execute SQL and return a DataFrame."""
    conn = psycopg2.connect(**DB_PARAMS)
    df = pd.read_sql(sql, conn)
    conn.close()
    return df


def save(fig, name):
    """Save figure to visualizations folder."""
    path = os.path.join(VIZ_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {name}.png")



# 1. Year-over-Year Revenue & Profit Growth

print("Generating charts...")
print("  [1/12] YoY Revenue & Profit")

df = run_query("""
    SELECT order_year AS year,
           SUM(sales) AS revenue,
           SUM(profit) AS profit,
           COUNT(DISTINCT order_id) AS orders
    FROM sales_data GROUP BY order_year ORDER BY order_year
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(df["year"].astype(str), df["revenue"], color=PALETTE[0], alpha=0.85, label="Revenue")
ax1.bar(df["year"].astype(str), df["profit"], color=PALETTE[2], alpha=0.85, label="Profit")
ax1.set_title("Revenue vs Profit by Year", fontsize=14, fontweight="bold")
ax1.set_xlabel("Year")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax1.legend()

ax2.plot(df["year"].astype(str), df["orders"], color=PALETTE[0], marker="o", linewidth=2.5, markersize=8)
ax2.set_title("Total Orders by Year", fontsize=14, fontweight="bold")
ax2.set_xlabel("Year")
for i, v in enumerate(df["orders"]):
    ax2.annotate(f"{v:,}", (df["year"].astype(str)[i], v), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
save(fig, "01_yoy_revenue_profit")



# 2. Monthly Revenue Trend (with moving average)

print("  [2/12] Monthly Trend")

df = run_query("""
    WITH monthly AS (
        SELECT order_year, order_month, SUM(sales) AS revenue
        FROM sales_data GROUP BY order_year, order_month
    )
    SELECT TO_CHAR(MAKE_DATE(order_year, order_month, 1), 'Mon YYYY') AS month_label,
           revenue,
           AVG(revenue) OVER (ORDER BY order_year, order_month
                              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma3,
           order_year, order_month
    FROM monthly ORDER BY order_year, order_month
""")

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(len(df)), df["revenue"], color=PALETTE[0], alpha=0.5, linewidth=1, label="Monthly")
ax.plot(range(len(df)), df["ma3"], color=PALETTE[3], linewidth=2.5, label="3-Month Avg")
ax.fill_between(range(len(df)), df["revenue"], alpha=0.1, color=PALETTE[0])
tick_pos = range(0, len(df), 3)
ax.set_xticks(list(tick_pos))
ax.set_xticklabels([df.iloc[i]["month_label"] for i in tick_pos], rotation=45, ha="right")
ax.set_title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.legend()
plt.tight_layout()
save(fig, "02_monthly_trend")



# 3. Sales by Region (pie + bar)

print("  [3/12] Regional Sales")

df = run_query("""
    SELECT region, SUM(sales) AS sales, SUM(profit) AS profit,
           COUNT(DISTINCT order_id) AS orders
    FROM sales_data GROUP BY region ORDER BY sales DESC
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.pie(df["sales"], labels=df["region"], autopct="%1.1f%%",
        colors=PALETTE[:4], startangle=90, textprops={"fontsize": 11})
ax1.set_title("Revenue Share by Region", fontsize=14, fontweight="bold")

bars = ax2.barh(df["region"][::-1], df["profit"][::-1], color=[PALETTE[2] if p > 0 else PALETTE[3] for p in df["profit"][::-1]])
ax2.set_title("Profit by Region", fontsize=14, fontweight="bold")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
for bar, val in zip(bars, df["profit"][::-1]):
    ax2.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
             f"${val:,.0f}", va="center", fontsize=10)
plt.tight_layout()
save(fig, "03_regional_sales")



# 4. Top 10 States by Revenue

print("  [4/12] Top States")

df = run_query("""
    SELECT state, region, SUM(sales) AS sales, SUM(profit) AS profit
    FROM sales_data GROUP BY state, region ORDER BY sales DESC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(df)-1, -1, -1), df["sales"], color=PALETTE[0], alpha=0.85)
ax.set_yticks(range(len(df)-1, -1, -1))
ax.set_yticklabels([f"{s} ({r})" for s, r in zip(df["state"], df["region"])])
ax.set_title("Top 10 States by Revenue", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
for bar, val in zip(bars, df["sales"]):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)
plt.tight_layout()
save(fig, "04_top_states")



# 5. Category & Sub-Category Performance

print("  [5/12] Category Performance")

df = run_query("""
    SELECT category, sub_category, SUM(sales) AS sales, SUM(profit) AS profit,
           ROUND(AVG(profit_margin), 1) AS avg_margin
    FROM sales_data GROUP BY category, sub_category ORDER BY sales DESC
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Sub-category sales
colors = [PALETTE[2] if p >= 0 else PALETTE[3] for p in df["profit"]]
ax1.barh(range(len(df)-1, -1, -1), df["sales"], color=colors, alpha=0.85)
ax1.set_yticks(range(len(df)-1, -1, -1))
ax1.set_yticklabels([f"{sc} ({c})" for c, sc in zip(df["category"], df["sub_category"])], fontsize=9)
ax1.set_title("Sales by Sub-Category (green=profit, red=loss)", fontsize=13, fontweight="bold")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))

# Category summary
cat_df = df.groupby("category").agg(sales=("sales", "sum"), profit=("profit", "sum")).reset_index()
x = np.arange(len(cat_df))
w = 0.35
ax2.bar(x - w/2, cat_df["sales"], w, color=PALETTE[0], label="Sales")
ax2.bar(x + w/2, cat_df["profit"], w, color=PALETTE[2], label="Profit")
ax2.set_xticks(x)
ax2.set_xticklabels(cat_df["category"])
ax2.set_title("Sales vs Profit by Category", fontsize=13, fontweight="bold")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax2.legend()
plt.tight_layout()
save(fig, "05_category_performance")



# 6. Top 10 Products by Revenue

print("  [6/12] Top Products")

df = run_query("""
    SELECT LEFT(product_name, 45) AS product_name, category,
           SUM(sales) AS revenue, SUM(profit) AS profit
    FROM sales_data GROUP BY product_name, category
    ORDER BY revenue DESC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(df)-1, -1, -1), df["revenue"], color=PALETTE[0], alpha=0.85)
ax.set_yticks(range(len(df)-1, -1, -1))
ax.set_yticklabels(df["product_name"], fontsize=9)
ax.set_title("Top 10 Products by Revenue", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
for bar, val in zip(bars, df["revenue"]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)
plt.tight_layout()
save(fig, "06_top_products")



# 7. Worst 10 Products (Loss Makers)

print("  [7/12] Worst Products")

df = run_query("""
    SELECT LEFT(product_name, 45) AS product_name,
           SUM(profit) AS total_loss, AVG(discount) AS avg_discount
    FROM sales_data GROUP BY product_name
    HAVING SUM(profit) < 0 ORDER BY total_loss ASC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(df)-1, -1, -1), df["total_loss"], color=PALETTE[3], alpha=0.85)
ax.set_yticks(range(len(df)-1, -1, -1))
ax.set_yticklabels(df["product_name"], fontsize=9)
ax.set_title("10 Biggest Loss-Making Products", fontsize=14, fontweight="bold", color=PALETTE[3])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.axvline(x=0, color="black", linewidth=0.8)
plt.tight_layout()
save(fig, "07_worst_products")



# 8. Customer Segments

print("  [8/12] Customer Segments")

df = run_query("""
    SELECT segment, COUNT(DISTINCT customer_id) AS customers,
           SUM(sales) AS sales, SUM(profit) AS profit
    FROM sales_data GROUP BY segment ORDER BY sales DESC
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.pie(df["sales"], labels=df["segment"], autopct="%1.1f%%",
        colors=PALETTE[:3], startangle=90, textprops={"fontsize": 11})
ax1.set_title("Revenue by Customer Segment", fontsize=14, fontweight="bold")

x = np.arange(len(df))
ax2.bar(x, df["customers"], color=PALETTE[4], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(df["segment"])
ax2.set_title("Number of Customers by Segment", fontsize=14, fontweight="bold")
for i, v in enumerate(df["customers"]):
    ax2.text(i, v + 5, str(v), ha="center", fontweight="bold")
plt.tight_layout()
save(fig, "08_customer_segments")



# 9. Discount Impact on Profitability

print("  [9/12] Discount Impact")

df = run_query("""
    SELECT discount_tier, COUNT(*) AS txns, SUM(profit) AS profit,
           ROUND(AVG(profit_margin), 1) AS avg_margin,
           SUM(is_loss) AS losses
    FROM sales_data GROUP BY discount_tier ORDER BY MIN(discount)
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = [PALETTE[2] if p >= 0 else PALETTE[3] for p in df["profit"]]
ax1.bar(df["discount_tier"], df["profit"], color=colors, alpha=0.85)
ax1.set_title("Total Profit by Discount Tier", fontsize=14, fontweight="bold")
ax1.set_ylabel("Profit ($)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax1.tick_params(axis="x", rotation=30)

ax2.bar(df["discount_tier"], df["losses"], color=PALETTE[3], alpha=0.85)
ax2.set_title("Loss Transactions by Discount Tier", fontsize=14, fontweight="bold")
ax2.set_ylabel("Number of Losses")
ax2.tick_params(axis="x", rotation=30)
plt.tight_layout()
save(fig, "09_discount_impact")



# 10. Shipping Performance

print("  [10/12] Shipping Performance")

df = run_query("""
    SELECT ship_mode, COUNT(*) AS shipments, AVG(shipping_days) AS avg_days,
           SUM(sales) AS sales
    FROM sales_data GROUP BY ship_mode ORDER BY avg_days
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(df["ship_mode"], df["avg_days"], color=PALETTE[4], alpha=0.85)
ax1.set_title("Avg Shipping Days by Mode", fontsize=14, fontweight="bold")
ax1.set_ylabel("Days")
for i, v in enumerate(df["avg_days"]):
    ax1.text(i, v + 0.1, f"{v:.1f}", ha="center", fontweight="bold")

ax2.barh(df["ship_mode"][::-1], df["sales"][::-1], color=PALETTE[0], alpha=0.85)
ax2.set_title("Revenue by Shipping Mode", fontsize=14, fontweight="bold")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
plt.tight_layout()
save(fig, "10_shipping_performance")



# 11. Seasonality Heatmap

print("  [11/12] Seasonality Heatmap")

df = run_query("""
    SELECT order_year,
           SUM(CASE WHEN order_month=1  THEN sales ELSE 0 END) AS Jan,
           SUM(CASE WHEN order_month=2  THEN sales ELSE 0 END) AS Feb,
           SUM(CASE WHEN order_month=3  THEN sales ELSE 0 END) AS Mar,
           SUM(CASE WHEN order_month=4  THEN sales ELSE 0 END) AS Apr,
           SUM(CASE WHEN order_month=5  THEN sales ELSE 0 END) AS May,
           SUM(CASE WHEN order_month=6  THEN sales ELSE 0 END) AS Jun,
           SUM(CASE WHEN order_month=7  THEN sales ELSE 0 END) AS Jul,
           SUM(CASE WHEN order_month=8  THEN sales ELSE 0 END) AS Aug,
           SUM(CASE WHEN order_month=9  THEN sales ELSE 0 END) AS Sep,
           SUM(CASE WHEN order_month=10 THEN sales ELSE 0 END) AS Oct,
           SUM(CASE WHEN order_month=11 THEN sales ELSE 0 END) AS Nov,
           SUM(CASE WHEN order_month=12 THEN sales ELSE 0 END) AS Dec
    FROM sales_data GROUP BY order_year ORDER BY order_year
""")

df = df.set_index("order_year")
df = df / 1000  # to thousands

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=1,
            ax=ax, cbar_kws={"label": "Revenue ($K)"})
ax.set_title("Monthly Revenue Heatmap by Year ($K)", fontsize=14, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
save(fig, "11_seasonality_heatmap")



# 12. Executive Dashboard (combined KPIs)

print("  [12/12] Executive Dashboard")

kpi = run_query("""
    SELECT
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT customer_id) AS customers,
        ROUND(SUM(sales)) AS revenue,
        ROUND(SUM(profit)) AS profit,
        ROUND((SUM(profit)/NULLIF(SUM(sales),0)*100)::NUMERIC, 1) AS margin,
        ROUND(AVG(sales)::NUMERIC, 2) AS avg_order,
        SUM(is_loss) AS losses,
        ROUND((SUM(is_loss)::NUMERIC/COUNT(*)*100), 1) AS loss_rate
    FROM sales_data
""")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("EXECUTIVE DASHBOARD — Superstore Sales (2014-2017)",
             fontsize=18, fontweight="bold", y=0.98)

kpis = [
    ("Total Revenue", f"${kpi['revenue'].iloc[0]:,.0f}", PALETTE[0]),
    ("Total Profit", f"${kpi['profit'].iloc[0]:,.0f}", PALETTE[2]),
    ("Profit Margin", f"{kpi['margin'].iloc[0]}%", PALETTE[4]),
    ("Total Orders", f"{kpi['total_orders'].iloc[0]:,}", PALETTE[0]),
    ("Avg Order Value", f"${kpi['avg_order'].iloc[0]:,.2f}", PALETTE[5]),
    ("Customers", f"{kpi['customers'].iloc[0]:,}", PALETTE[4]),
    ("Loss Transactions", f"{kpi['losses'].iloc[0]:,}", PALETTE[3]),
    ("Loss Rate", f"{kpi['loss_rate'].iloc[0]}%", PALETTE[3]),
]

for ax, (label, value, color) in zip(axes.flat, kpis):
    ax.text(0.5, 0.65, value, transform=ax.transAxes, ha="center", va="center",
            fontsize=28, fontweight="bold", color=color)
    ax.text(0.5, 0.25, label, transform=ax.transAxes, ha="center", va="center",
            fontsize=13, color="#555555")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.patch.set_facecolor("#f8f9fa")
    ax.patch.set_alpha(0.7)

plt.tight_layout(rect=[0, 0, 1, 0.95])
save(fig, "12_executive_dashboard")


print(f"\nAll charts saved to: {VIZ_DIR}/")
print(f"Total: 12 visualization files generated.")


## **Interactive Dashboards**

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_data.csv")
OUT_DIR = os.path.join(PROJECT_ROOT, "visualizations", "advanced")
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading data...")
df = pd.read_csv(DATA_PATH, encoding="utf-8")
df.columns = [c.strip().replace(" ", "_").replace("-", "_").lower() for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)
print(f"Loaded {len(df):,} rows\n")

COLORS = ["#2563eb", "#f59e0b", "#10b981", "#ef4444", "#8b5cf6", "#06b6d4"]
TEMPLATE = "plotly_white"


# Dashboard 1: Sales & Revenue Overview

print("Generating Dashboard 1: Sales & Revenue Overview...")

monthly = df.groupby("year_month").agg(
    revenue=("sales", "sum"), profit=("profit", "sum"),
    orders=("order_id", "nunique"), transactions=("sales", "count")
).reset_index()

fig1 = make_subplots(rows=2, cols=2,
    subplot_titles=("Monthly Revenue Trend", "Revenue by Region",
                    "Revenue by Category Over Time", "Discount Impact on Profit"),
    specs=[[{"type": "scatter"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "scatter"}]])

# Monthly trend
fig1.add_trace(go.Scatter(x=monthly["year_month"], y=monthly["revenue"],
    mode="lines+markers", name="Revenue", line=dict(color=COLORS[0], width=2),
    marker=dict(size=5)), row=1, col=1)
fig1.add_trace(go.Scatter(x=monthly["year_month"], y=monthly["profit"],
    mode="lines+markers", name="Profit", line=dict(color=COLORS[2], width=2),
    marker=dict(size=5)), row=1, col=1)

# Revenue by region
region_data = df.groupby("region")["sales"].sum().reset_index()
fig1.add_trace(go.Pie(labels=region_data["region"], values=region_data["sales"],
    marker=dict(colors=COLORS[:4]), hole=0.4, textinfo="label+percent"), row=1, col=2)

# Revenue by category over time
cat_monthly = df.groupby(["year_month", "category"])["sales"].sum().reset_index()
for i, cat in enumerate(df["category"].unique()):
    cat_df = cat_monthly[cat_monthly["category"] == cat]
    fig1.add_trace(go.Bar(x=cat_df["year_month"], y=cat_df["sales"],
        name=cat, marker_color=COLORS[i]), row=2, col=1)

# Discount vs profit
disc_data = df.groupby("discount").agg(avg_profit=("profit", "mean"), count=("sales", "count")).reset_index()
fig1.add_trace(go.Scatter(x=disc_data["discount"] * 100, y=disc_data["avg_profit"],
    mode="markers+lines", name="Avg Profit", marker=dict(size=disc_data["count"] / 20, color=COLORS[3]),
    line=dict(color=COLORS[3], width=2)), row=2, col=2)

fig1.update_layout(height=900, title_text="Sales & Revenue Dashboard", template=TEMPLATE,
    title_font_size=20, showlegend=True)
fig1.write_html(os.path.join(OUT_DIR, "dashboard_sales.html"))
print("  Saved: dashboard_sales.html")



# Dashboard 2: Customer Analytics

print("Generating Dashboard 2: Customer Analytics...")

snapshot = df["order_date"].max() + pd.Timedelta(days=1)
rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (snapshot - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("sales", "sum")
).reset_index()

# Add customer name
cust_names = df[["customer_id", "customer_name"]].drop_duplicates()
rfm = rfm.merge(cust_names, on="customer_id")

fig2 = make_subplots(rows=2, cols=2,
    subplot_titles=("Customer RFM Scatter (Frequency vs Monetary)", "Customer Segments by Revenue",
                    "Order Frequency Distribution", "Customer Revenue Distribution"),
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "histogram"}]])

fig2.add_trace(go.Scatter(x=rfm["frequency"], y=rfm["monetary"], mode="markers",
    marker=dict(size=8, color=rfm["recency"], colorscale="RdYlGn", showscale=True,
                colorbar=dict(title="Recency (days)")),
    text=rfm["customer_name"], hovertemplate="%{text}<br>Freq: %{x}<br>Monetary: $%{y:,.0f}<extra></extra>"),
    row=1, col=1)

seg_revenue = df.groupby("segment").agg(
    revenue=("sales", "sum"), customers=("customer_id", "nunique")
).reset_index().sort_values("revenue", ascending=True)
fig2.add_trace(go.Bar(x=seg_revenue["revenue"], y=seg_revenue["segment"],
    orientation="h", marker_color=COLORS[:3],
    text=seg_revenue["revenue"].apply(lambda x: f"${x:,.0f}"), textposition="outside"),
    row=1, col=2)

fig2.add_trace(go.Histogram(x=rfm["frequency"], nbinsx=30, marker_color=COLORS[0], name="Frequency"),
    row=2, col=1)
fig2.add_trace(go.Histogram(x=rfm["monetary"], nbinsx=50, marker_color=COLORS[2], name="Monetary"),
    row=2, col=2)

fig2.update_layout(height=900, title_text="Customer Analytics Dashboard", template=TEMPLATE,
    title_font_size=20, showlegend=False)
fig2.write_html(os.path.join(OUT_DIR, "dashboard_customers.html"))
print("  Saved: dashboard_customers.html")



# Dashboard 3: Product & Profitability

print("Generating Dashboard 3: Product & Profitability...")

prod = df.groupby(["product_name", "category", "sub_category"]).agg(
    total_sales=("sales", "sum"), total_profit=("profit", "sum"),
    units_sold=("quantity", "sum"), avg_discount=("discount", "mean")
).reset_index()

fig3 = make_subplots(rows=2, cols=2,
    subplot_titles=("Top 15 Products by Revenue", "Profit by Sub-Category",
                    "Category Performance (Treemap)", "Profit Margin Distribution"),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "treemap"}, {"type": "histogram"}]])

top15 = prod.nlargest(15, "total_sales")
fig3.add_trace(go.Bar(x=top15["total_sales"], y=top15["product_name"].apply(lambda x: x[:40]),
    orientation="h", marker_color=COLORS[0], name="Revenue",
    text=top15["total_sales"].apply(lambda x: f"${x:,.0f}"), textposition="outside"),
    row=1, col=1)

sub_cat = df.groupby("sub_category")["profit"].sum().sort_values().reset_index()
colors_bar = [COLORS[2] if v >= 0 else COLORS[3] for v in sub_cat["profit"]]
fig3.add_trace(go.Bar(x=sub_cat["profit"], y=sub_cat["sub_category"],
    orientation="h", marker_color=colors_bar, name="Profit"),
    row=1, col=2)

fig3.add_trace(go.Treemap(labels=prod["sub_category"], parents=prod["category"],
    values=prod["total_sales"], marker=dict(colors=COLORS)),
    row=2, col=1)

margin_data = df["profit_margin"].clip(-100, 100)
fig3.add_trace(go.Histogram(x=margin_data, nbinsx=60, marker_color=COLORS[4], name="Margin %"),
    row=2, col=2)

fig3.update_layout(height=900, title_text="Product & Profitability Dashboard", template=TEMPLATE,
    title_font_size=20, showlegend=False)
fig3.write_html(os.path.join(OUT_DIR, "dashboard_products.html"))
print("  Saved: dashboard_products.html")



# Dashboard 4: Geographic & Time Analysis

print("Generating Dashboard 4: Geographic & Time Analysis...")

state_data = df.groupby(["state", "region"]).agg(
    revenue=("sales", "sum"), profit=("profit", "sum"),
    orders=("order_id", "nunique")
).reset_index()

fig4 = make_subplots(rows=2, cols=2,
    subplot_titles=("Revenue by State (Top 20)", "Revenue Heatmap: Year x Month",
                    "Shipping Days by Region", "Day-of-Week Sales Pattern"),
    specs=[[{"type": "bar"}, {"type": "heatmap"}],
           [{"type": "bar"}, {"type": "scatter"}]])

top20_states = state_data.nlargest(20, "revenue")
fig4.add_trace(go.Bar(x=top20_states["revenue"], y=top20_states["state"],
    orientation="h", marker_color=COLORS[0],
    text=top20_states["revenue"].apply(lambda x: f"${x:,.0f}"), textposition="outside"),
    row=1, col=1)

# Heatmap
heatmap_data = df.groupby(["order_year", "order_month"])["sales"].sum().reset_index()
pivot = heatmap_data.pivot_table(index="order_year", columns="order_month", values="sales")
fig4.add_trace(go.Heatmap(z=pivot.values, x=[f"M{m}" for m in pivot.columns],
    y=pivot.index.tolist(), colorscale="YlOrRd", text=np.round(pivot.values / 1000, 0),
    texttemplate="%{text}K"), row=1, col=2)

# Shipping by region
ship_region = df.groupby("region")["shipping_days"].mean().sort_values().reset_index()
fig4.add_trace(go.Bar(x=ship_region["region"], y=ship_region["shipping_days"],
    marker_color=COLORS[4], name="Avg Shipping Days"), row=2, col=1)

# Day of week pattern
dow = df.groupby("order_dayname").agg(
    revenue=("sales", "sum"), orders=("order_id", "nunique")
).reset_index()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow["order"] = dow["order_dayname"].map({d: i for i, d in enumerate(day_order)})
dow = dow.sort_values("order")
fig4.add_trace(go.Scatter(x=dow["order_dayname"], y=dow["revenue"],
    mode="lines+markers", line=dict(color=COLORS[1], width=3), marker=dict(size=10)),
    row=2, col=2)

fig4.update_layout(height=900, title_text="Geographic & Temporal Dashboard", template=TEMPLATE,
    title_font_size=20, showlegend=False)
fig4.write_html(os.path.join(OUT_DIR, "dashboard_geotemporal.html"))
print("  Saved: dashboard_geotemporal.html")


print(f"\nAll 4 interactive dashboards saved to: {OUT_DIR}/")
print("Open the .html files in your browser to interact with the charts.")


## **Streamlit Dashboard**

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st
from datetime import date

warnings.filterwarnings("ignore")

# Page config
st.set_page_config(
    page_title="Superstore Sales Dashboard",
    page_icon=":bar_chart:",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Colors
BLUE, GREEN, AMBER, RED, PURPLE, CYAN = "#2563eb", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6", "#06b6d4"
COLORS = [BLUE, AMBER, GREEN, RED, PURPLE, CYAN]

# Custom CSS
st.markdown("""
<style>
    .main { background-color: #f5f5f5; }
    div[data-testid="stMetric"] {
        background: #111111;
        padding: 18px 15px;
        border-radius: 10px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.3);
    }
    div[data-testid="stMetric"] label {
        font-size: 0.85rem;
        color: #aaaaaa !important;
    }
    div[data-testid="stMetric"] div[data-testid="stMetricValue"] {
        color: #ffffff !important;
    }
    div[data-testid="stMetric"] div[data-testid="stMetricDelta"] {
        color: #aaaaaa !important;
    }
    /* Green for positive deltas */
    div[data-testid="stMetric"] svg[data-testid="stMetricDeltaIcon"] ~ div {
        color: #10b981 !important;
    }
    div[data-testid="stSidebar"] { background: #1b2838; color: white; }
    div[data-testid="stSidebar"] * { color: white !important; }
    h1 { color: #1b3a5c; }
</style>
""", unsafe_allow_html=True)


# Data Loading
@st.cache_data
def load_data():
    path = os.path.join(os.path.abspath('.'),
                        "..", "data", "processed", "clean_data.csv")
    df = pd.read_csv(path, encoding="utf-8")
    df.columns = [c.strip().replace(" ", "_").replace("-", "_").lower() for c in df.columns]
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["ship_date"] = pd.to_datetime(df["ship_date"])
    return df

df = load_data()


# Sidebar Filters
st.sidebar.markdown("## Filters")
st.sidebar.markdown("---")

# Year filter
years = sorted(df["order_year"].unique())
selected_years = st.sidebar.multiselect("Year", years, default=years)

# Category filter
categories = sorted(df["category"].unique())
selected_categories = st.sidebar.multiselect("Category", categories, default=categories)

# Region filter
regions = sorted(df["region"].unique())
selected_regions = st.sidebar.multiselect("Region", regions, default=regions)

# Segment filter
segments = sorted(df["segment"].unique())
selected_segments = st.sidebar.multiselect("Segment", segments, default=segments)

# Date range
min_date = df["order_date"].min().date()
max_date = df["order_date"].max().date()
date_range = st.sidebar.slider("Date Range", min_value=min_date, max_value=max_date,
                                value=(min_date, max_date))

# Apply filters
mask = (
    df["order_year"].isin(selected_years) &
    df["category"].isin(selected_categories) &
    df["region"].isin(selected_regions) &
    df["segment"].isin(selected_segments) &
    (df["order_date"].dt.date >= date_range[0]) &
    (df["order_date"].dt.date <= date_range[1])
)
fdf = df[mask]  # filtered dataframe

st.sidebar.markdown("---")
st.sidebar.markdown(f"**{len(fdf):,}** transactions | **{fdf['order_id'].nunique():,}** orders")
st.sidebar.markdown(f"Data source: `clean_data.csv` (9,986 rows)")


# Helper Functions
def kpi_row(data):
    """Render KPI metric cards."""
    total_sales = data["sales"].sum()
    total_profit = data["profit"].sum()
    total_orders = data["order_id"].nunique()
    total_customers = data["customer_id"].nunique()
    avg_order = total_sales / total_orders if total_orders > 0 else 0
    margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
    loss_count = data["is_loss"].sum()

    c1, c2, c3, c4, c5, c6, c7 = st.columns(7)
    c1.metric("Total Sales", f"${total_sales:,.0f}")
    c2.metric("Total Profit", f"${total_profit:,.0f}",
              delta=f"{total_profit/total_sales*100:.1f}% margin" if total_sales > 0 else "0%")
    c3.metric("Total Orders", f"{total_orders:,}")
    c4.metric("Customers", f"{total_customers:,}")
    c5.metric("Avg Order Value", f"${avg_order:,.2f}")
    c6.metric("Profit Margin", f"{margin:.1f}%")
    c7.metric("Loss Transactions", f"{int(loss_count):,}",
              delta=f"{loss_count/len(data)*100:.1f}% loss rate" if len(data) > 0 else "0%",
              delta_color="inverse")


def monthly_agg(data):
    """Aggregate data by year-month."""
    m = data.copy()
    m["ym"] = m["order_date"].dt.to_period("M").astype(str)
    m["ym_date"] = m["order_date"].dt.to_period("M").dt.to_timestamp()
    g = m.groupby(["ym", "ym_date"]).agg(
        revenue=("sales", "sum"), profit=("profit", "sum"),
        orders=("order_id", "nunique"), transactions=("sales", "count")
    ).reset_index().sort_values("ym_date")
    g["margin_pct"] = (g["profit"] / g["revenue"] * 100).round(1)
    g["ma3"] = g["revenue"].rolling(3, min_periods=1).mean()
    g["ma6"] = g["revenue"].rolling(6, min_periods=1).mean()
    return g


# Title
st.markdown("# Superstore Sales Dashboard")
st.markdown("*Performance analytics for FY 2014–2017 | Data source: PostgreSQL working_database*")
st.markdown("---")


# Tab Navigation
tab1, tab2, tab3, tab4, tab5 = st.tabs(
    ["Executive Overview", "Sales Trends", "Products", "Customers", "Geography"])


# TAB 1: EXECUTIVE OVERVIEW

with tab1:
    kpi_row(fdf)
    st.markdown("")

    # Row 1: Monthly trend + Category donut
    c1, c2 = st.columns([2, 1])
    with c1:
        mg = monthly_agg(fdf)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["revenue"], mode="lines+markers",
                                  name="Revenue", line=dict(color=BLUE, width=2)))
        fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["profit"], mode="lines+markers",
                                  name="Profit", line=dict(color=GREEN, width=2)))
        fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["ma3"], mode="lines",
                                  name="3M Avg", line=dict(color=RED, width=1.5, dash="dash")))
        fig.update_layout(title="Monthly Revenue & Profit Trend", template="plotly_white",
                          height=380, margin=dict(l=20, r=20, t=50, b=20))
        fig.update_yaxes(tickprefix="$", tickformat=".0f")
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        cat_data = fdf.groupby("category")["sales"].sum().reset_index()
        fig = px.pie(cat_data, values="sales", names="category", hole=0.55,
                     color="category", color_discrete_sequence=COLORS[:3])
        fig.update_layout(title="Revenue by Category", template="plotly_white",
                          height=380, margin=dict(l=20, r=20, t=50, b=20),
                          showlegend=False)
        fig.update_traces(textinfo="percent+label", textfont_size=13)
        st.plotly_chart(fig, use_container_width=True)

    # Row 2: Region bar + Segment bar
    c1, c2 = st.columns(2)
    with c1:
        reg = fdf.groupby("region").agg(sales=("sales", "sum"), profit=("profit", "sum")).reset_index()
        reg = reg.sort_values("sales", ascending=True)
        fig = go.Figure()
        fig.add_trace(go.Bar(y=reg["region"], x=reg["sales"], name="Sales",
                             orientation="h", marker_color=BLUE))
        fig.add_trace(go.Bar(y=reg["region"], x=reg["profit"], name="Profit",
                             orientation="h", marker_color=GREEN))
        fig.update_layout(title="Sales vs Profit by Region", template="plotly_white",
                          barmode="group", height=350, margin=dict(l=20, r=20, t=50, b=20))
        fig.update_xaxes(tickprefix="$", tickformat=".0f")
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        seg = fdf.groupby("segment").agg(
            sales=("sales", "sum"), customers=("customer_id", "nunique")).reset_index()
        seg = seg.sort_values("sales", ascending=True)
        fig = go.Figure()
        fig.add_trace(go.Bar(y=seg["segment"], x=seg["sales"], name="Revenue",
                             orientation="h", marker_color=PURPLE,
                             text=seg["sales"].apply(lambda x: f"${x:,.0f}"), textposition="outside"))
        fig.update_layout(title="Revenue by Customer Segment", template="plotly_white",
                          height=350, margin=dict(l=20, r=20, t=50, b=20), showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

    # Row 3: Ship mode + Discount impact
    c1, c2 = st.columns(2)
    with c1:
        ship = fdf.groupby("ship_mode").agg(
            txns=("sales", "count"), sales=("sales", "sum"),
            avg_days=("shipping_days", "mean")).reset_index().sort_values("sales", ascending=True)
        fig = px.bar(ship, x="sales", y="ship_mode", orientation="h",
                     color="avg_days", color_continuous_scale="Blues",
                     labels={"sales": "Revenue", "ship_mode": "Ship Mode", "avg_days": "Avg Days"})
        fig.update_layout(title="Revenue by Shipping Mode", template="plotly_white",
                          height=350, margin=dict(l=20, r=20, t=50, b=20))
        fig.update_xaxes(tickprefix="$", tickformat=".0f")
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        disc = fdf.groupby("discount_tier").agg(
            sales=("sales", "sum"), profit=("profit", "sum"),
            loss_rate=("is_loss", "mean")).reset_index()
        disc["loss_rate"] = (disc["loss_rate"] * 100).round(1)
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Bar(x=disc["discount_tier"], y=disc["profit"], name="Profit",
                             marker_color=[GREEN if p >= 0 else RED for p in disc["profit"]]),
                      secondary_y=False)
        fig.add_trace(go.Scatter(x=disc["discount_tier"], y=disc["loss_rate"],
                                  name="Loss Rate %", line=dict(color=RED, width=2)),
                      secondary_y=True)
        fig.update_layout(title="Discount Impact on Profitability", template="plotly_white",
                          height=350, margin=dict(l=20, r=20, t=50, b=20))
        fig.update_yaxes(title_text="Profit ($)", secondary_y=False)
        fig.update_yaxes(title_text="Loss Rate %", secondary_y=True)
        st.plotly_chart(fig, use_container_width=True)



# TAB 2: SALES TRENDS

with tab2:
    mg = monthly_agg(fdf)

    # YoY Growth
    st.markdown("### Year-over-Year Performance")
    yoy = fdf.groupby("order_year").agg(
        revenue=("sales", "sum"), profit=("profit", "sum"),
        orders=("order_id", "nunique")).reset_index()
    yoy["rev_growth"] = yoy["revenue"].pct_change() * 100
    yoy["profit_growth"] = yoy["profit"].pct_change() * 100

    c1, c2, c3, c4 = st.columns(4)
    for i, yr in enumerate(yoy["order_year"]):
        row = yoy[yoy["order_year"] == yr].iloc[0]
        col = [c1, c2, c3, c4][i]
        col.metric(f"{int(yr)}", f"${row['revenue']:,.0f}",
                   delta=f"{row['rev_growth']:.1f}%" if pd.notna(row['rev_growth']) else "—")

    st.markdown("---")

    # Moving averages chart
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["revenue"], mode="lines",
                              name="Monthly Revenue", line=dict(color=BLUE, width=1), opacity=0.4,
                              fill="tozeroy", fillcolor="rgba(37,99,235,0.05)"))
    fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["ma3"], mode="lines",
                              name="3-Month MA", line=dict(color=RED, width=2.5)))
    fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["ma6"], mode="lines",
                              name="6-Month MA", line=dict(color=AMBER, width=2.5)))
    # Forecast annotation
    fig.update_layout(title="Revenue Trend with Moving Averages", template="plotly_white",
                      height=420, margin=dict(l=20, r=20, t=50, b=20))
    fig.update_yaxes(tickprefix="$", tickformat=".0f")
    st.plotly_chart(fig, use_container_width=True)

    # Dual axis: Revenue bars + Profit margin line
    st.markdown("### Revenue & Profit Margin")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=mg["ym_date"], y=mg["revenue"], name="Revenue",
                         marker_color=BLUE, opacity=0.6), secondary_y=False)
    fig.add_trace(go.Scatter(x=mg["ym_date"], y=mg["margin_pct"], name="Margin %",
                             line=dict(color=GREEN, width=2.5)), secondary_y=True)
    fig.update_layout(template="plotly_white", height=380, margin=dict(l=20, r=20, t=30, b=20))
    fig.update_yaxes(title_text="Revenue ($)", tickprefix="$", secondary_y=False)
    fig.update_yaxes(title_text="Margin %", ticksuffix="%", secondary_y=True)
    st.plotly_chart(fig, use_container_width=True)

    # Seasonality heatmap
    st.markdown("### Seasonality Heatmap (Revenue by Year × Month)")
    heatmap = fdf.groupby(["order_year", "order_month"])["sales"].sum().reset_index()
    pivot = heatmap.pivot_table(index="order_year", columns="order_month", values="sales")
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    fig = go.Figure(go.Heatmap(
        z=pivot.values / 1000, x=month_names, y=[int(y) for y in pivot.index],
        colorscale="YlOrRd", text=np.round(pivot.values / 1000, 1),
        texttemplate="%{text}K", hovertemplate="Year %{y}, %{x}: $%{text}K<extra></extra>"))
    fig.update_layout(title="Monthly Revenue Heatmap ($K)", template="plotly_white",
                      height=320, margin=dict(l=20, r=20, t=50, b=20))
    st.plotly_chart(fig, use_container_width=True)



# TAB 3: PRODUCTS

with tab3:
    prod = fdf.groupby(["product_name", "category", "sub_category"]).agg(
        sales=("sales", "sum"), profit=("profit", "sum"),
        units=("quantity", "sum"), transactions=("sales", "count")
    ).reset_index()

    # Top 10 Products
    c1, c2 = st.columns(2)
    with c1:
        top10 = prod.nlargest(10, "sales")
        fig = px.bar(top10, x="sales", y=top10["product_name"].apply(lambda x: x[:40]),
                     orientation="h", color="category", color_discrete_sequence=COLORS[:3],
                     labels={"sales": "Revenue", "product_name": ""})
        fig.update_layout(title="Top 10 Products by Revenue", template="plotly_white",
                          height=450, margin=dict(l=20, r=40, t=50, b=20), showlegend=False,
                          yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        # Sub-category performance
        subcat = fdf.groupby("sub_category").agg(
            sales=("sales", "sum"), profit=("profit", "sum")).reset_index()
        subcat = subcat.sort_values("profit", ascending=True)
        fig = px.bar(subcat, x="profit", y="sub_category", orientation="h",
                     color="profit", color_continuous_scale=["#ef4444", "#fbbf24", "#10b981"],
                     labels={"profit": "Profit", "sub_category": ""})
        fig.update_layout(title="Profit by Sub-Category", template="plotly_white",
                          height=450, margin=dict(l=20, r=40, t=50, b=20),
                          coloraxis_showscale=False)
        fig.add_vline(x=0, line_color="black", line_width=1)
        st.plotly_chart(fig, use_container_width=True)

    # Product scatter matrix
    st.markdown("### Product Performance Matrix")
    fig = px.scatter(prod, x="sales", y="profit", size="units", color="category",
                     hover_data=["product_name"], color_discrete_sequence=COLORS[:3],
                     labels={"sales": "Total Sales ($)", "profit": "Total Profit ($)"})
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=prod["sales"].median(), line_dash="dash", line_color="gray", opacity=0.5)
    fig.update_layout(title="Sales vs Profit (bubble size = units sold)", template="plotly_white",
                      height=450, margin=dict(l=20, r=20, t=50, b=20))
    st.plotly_chart(fig, use_container_width=True)

    # Downloadable table
    st.markdown("### Product Data Table")
    st.dataframe(prod.sort_values("sales", ascending=False).head(50),
                 use_container_width=True, height=400)
    st.download_button("Download Product Data (CSV)", prod.to_csv(index=False),
                       "product_analysis.csv", "text/csv")

# TAB 4: CUSTOMERS

with tab4:
    cust = fdf.groupby(["customer_id", "customer_name", "segment"]).agg(
        orders=("order_id", "nunique"), sales=("sales", "sum"),
        profit=("profit", "sum"), first_order=("order_date", "min"),
        last_order=("order_date", "max")
    ).reset_index()
    cust["margin"] = (cust["profit"] / cust["sales"] * 100).round(1)
    cust["avg_order_value"] = (cust["sales"] / cust["orders"]).round(2)

    # Top 10 Customers
    c1, c2 = st.columns(2)
    with c1:
        top_cust = cust.nlargest(10, "sales")
        fig = px.bar(top_cust, x="sales", y="customer_name", orientation="h",
                     color="segment", color_discrete_sequence=COLORS[:3],
                     labels={"sales": "Revenue", "customer_name": ""})
        fig.update_layout(title="Top 10 Customers by Revenue", template="plotly_white",
                          height=450, margin=dict(l=20, r=40, t=50, b=20),
                          yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        # Customer distribution
        fig = go.Figure()
        fig.add_trace(go.Histogram(x=cust["sales"], nbinsx=50, name="Revenue",
                                   marker_color=BLUE, opacity=0.7))
        fig.update_layout(title="Customer Revenue Distribution", template="plotly_white",
                          height=450, margin=dict(l=20, r=20, t=50, b=20),
                          xaxis_title="Total Revenue ($)", yaxis_title="Number of Customers")
        st.plotly_chart(fig, use_container_width=True)

    # Customer frequency vs monetary scatter
    st.markdown("### Customer Frequency vs Monetary Value")
    # Use absolute profit for bubble size (marker size must be non-negative)
    cust["abs_profit"] = cust["profit"].abs().clip(lower=1)
    fig = px.scatter(cust, x="orders", y="sales", size="abs_profit",
                     color="segment", hover_data=["customer_name", "margin", "profit"],
                     color_discrete_sequence=COLORS[:3],
                     labels={"orders": "Number of Orders", "sales": "Total Revenue ($)"})
    fig.update_layout(template="plotly_white", height=450,
                      margin=dict(l=20, r=20, t=30, b=20))
    st.plotly_chart(fig, use_container_width=True)

    # Repeat vs One-time
    st.markdown("### Customer Loyalty Tiers")
    cust["tier"] = pd.cut(cust["orders"], bins=[0, 1, 5, 10, 100],
                          labels=["One-time", "Repeat (2-5)", "Loyal (6-10)", "VIP (11+)"])
    tier_data = cust.groupby("tier", observed=True).agg(
        customers=("customer_id", "count"), revenue=("sales", "sum")).reset_index()
    c1, c2 = st.columns(2)
    with c1:
        fig = px.bar(tier_data, x="tier", y="customers", color="tier",
                     color_discrete_sequence=COLORS[:4])
        fig.update_layout(title="Customer Count by Tier", template="plotly_white",
                          height=350, showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        fig = px.bar(tier_data, x="tier", y="revenue", color="tier",
                     color_discrete_sequence=COLORS[:4])
        fig.update_layout(title="Revenue by Customer Tier", template="plotly_white",
                          height=350, showlegend=False)
        fig.update_yaxes(tickprefix="$", tickformat=".0f")
        st.plotly_chart(fig, use_container_width=True)

    # Downloadable table
    st.dataframe(cust.sort_values("sales", ascending=False).head(50),
                 use_container_width=True, height=400)
    st.download_button("Download Customer Data (CSV)", cust.to_csv(index=False),
                       "customer_analysis.csv", "text/csv")



# TAB 5: GEOGRAPHY

with tab5:
    geo = fdf.groupby(["state", "region", "city"]).agg(
        sales=("sales", "sum"), profit=("profit", "sum"),
        orders=("order_id", "nunique"), customers=("customer_id", "nunique")
    ).reset_index()
    geo["margin"] = (geo["profit"] / geo["sales"] * 100).round(1)

    # State map
    st.markdown("### Sales by State")
    state_data = fdf.groupby(["state", "region"]).agg(
        sales=("sales", "sum"), profit=("profit", "sum")).reset_index()

    fig = px.choropleth(state_data, locations=[s[:2].upper() for s in state_data["state"]],
                        locationmode="USA-states", scope="usa", color="sales",
                        color_continuous_scale="Blues", hover_name="state",
                        hover_data={"sales": ":,.0f", "profit": ":,.0f", "region": True},
                        labels={"sales": "Revenue"})
    fig.update_layout(title="Revenue by State", template="plotly_white",
                      height=500, margin=dict(l=20, r=20, t=50, b=20))
    st.plotly_chart(fig, use_container_width=True)

    # Top 15 States
    c1, c2 = st.columns(2)
    with c1:
        top_states = state_data.nlargest(15, "sales")
        fig = px.bar(top_states, x="sales", y="state", orientation="h",
                     color="region", color_discrete_sequence=COLORS[:4],
                     labels={"sales": "Revenue", "state": ""})
        fig.update_layout(title="Top 15 States by Revenue", template="plotly_white",
                          height=500, margin=dict(l=20, r=40, t=50, b=20),
                          yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        # Region breakdown
        reg_data = fdf.groupby(["region", "category"])["sales"].sum().reset_index()
        fig = px.bar(reg_data, x="region", y="sales", color="category",
                     color_discrete_sequence=COLORS[:3], barmode="stack",
                     labels={"sales": "Revenue", "region": "Region"})
        fig.update_layout(title="Revenue by Region & Category", template="plotly_white",
                          height=500, margin=dict(l=20, r=20, t=50, b=20))
        fig.update_yaxes(tickprefix="$", tickformat=".0f")
        st.plotly_chart(fig, use_container_width=True)

    # Profit map
    st.markdown("### Profitability by State")
    fig = px.choropleth(state_data, locations=[s[:2].upper() for s in state_data["state"]],
                        locationmode="USA-states", scope="usa", color="profit",
                        color_continuous_scale="RdYlGn", hover_name="state",
                        hover_data={"profit": ":,.0f", "sales": ":,.0f"},
                        labels={"profit": "Profit"})
    fig.update_layout(title="Profit by State (Green = Profit, Red = Loss)",
                      template="plotly_white", height=500,
                      margin=dict(l=20, r=20, t=50, b=20))
    st.plotly_chart(fig, use_container_width=True)

    # Downloadable table
    st.dataframe(geo.sort_values("sales", ascending=False).head(50),
                 use_container_width=True, height=400)
    st.download_button("Download Geographic Data (CSV)", geo.to_csv(index=False),
                       "geography_analysis.csv", "text/csv")


# Footer
st.markdown("---")
st.markdown(
    f"<div style='text-align:center; color:#999; font-size:0.8rem;'>"
    f"Superstore Sales Dashboard | Data: clean_data.csv (9,986 transactions, "
    f"2014-2017) | Built with Streamlit + Plotly | "
    f"Showing {len(fdf):,} filtered records</div>",
    unsafe_allow_html=True
)

## **Export HTML Dashboard**

In [ ]:
"""
Export the dashboard as a standalone interactive HTML file.
Run:  python scripts/export_html.py
Output: dashboard.html (open in any browser)
"""

import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_data.csv")
OUT_PATH = os.path.join(PROJECT_ROOT, "dashboard.html")
TEMPLATE = os.path.join(os.path.abspath('.'), "dashboard_template.html")

BLUE = "#2563eb"
GREEN = "#10b981"
AMBER = "#f59e0b"
RED = "#ef4444"
PURPLE = "#8b5cf6"

print("Loading data...")
df = pd.read_csv(DATA_PATH, encoding="utf-8")
df.columns = [c.strip().replace(" ", "_").replace("-", "_").lower() for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"])

# KPIs
total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
total_orders = df["order_id"].nunique()
total_customers = df["customer_id"].nunique()
avg_order = total_sales / total_orders
margin = total_profit / total_sales * 100
loss_count = int(df["is_loss"].sum())

# YoY
yoy = df.groupby("order_year").agg(revenue=("sales", "sum")).reset_index()
yoy["growth"] = yoy["revenue"].pct_change() * 100

# Monthly
df_m = df.copy()
df_m["ym_date"] = df_m["order_date"].dt.to_period("M").dt.to_timestamp()
mg = df_m.groupby("ym_date").agg(
    revenue=("sales", "sum"), profit=("profit", "sum"),
    orders=("order_id", "nunique")).reset_index().sort_values("ym_date")
mg["margin_pct"] = (mg["profit"] / mg["revenue"] * 100).round(1)
mg["ma3"] = mg["revenue"].rolling(3, min_periods=1).mean()

# Category
cat = df.groupby("category")["sales"].sum().reset_index()
# Region
reg = df.groupby("region").agg(sales=("sales", "sum"), profit=("profit", "sum")).reset_index().sort_values("sales")
# Top products
prod = df.groupby(["product_name", "category"]).agg(sales=("sales", "sum")).reset_index().nlargest(10, "sales")
# Top customers
cust = df.groupby(["customer_name", "segment"]).agg(sales=("sales", "sum")).reset_index().nlargest(10, "sales")
# Sub-category
subcat = df.groupby("sub_category").agg(profit=("profit", "sum")).reset_index().sort_values("profit")
# Heatmap
heatmap = df.groupby(["order_year", "order_month"])["sales"].sum().reset_index()
pivot = heatmap.pivot_table(index="order_year", columns="order_month", values="sales")
# Top states
state = df.groupby(["state", "region"]).agg(sales=("sales", "sum")).reset_index().nlargest(15, "sales")
# Discount
disc = df.groupby("discount_tier").agg(profit=("profit", "sum"), loss_rate=("is_loss", "mean")).reset_index()
disc["loss_rate"] = (disc["loss_rate"] * 100).round(1)
# Scatter
prod_all = df.groupby(["product_name", "category"]).agg(sales=("sales", "sum"), profit=("profit", "sum"), units=("quantity", "sum")).reset_index()
# US map
state_map = df.groupby(["state", "region"]).agg(sales=("sales", "sum"), profit=("profit", "sum")).reset_index()
# Customer histogram
cust_hist = df.groupby("customer_id").agg(sales=("sales", "sum")).reset_index()["sales"].round(0).tolist()

# YoY cards
yoy_html = ""
for _, row in yoy.iterrows():
    g = row["growth"]
    gs = f"{g:.1f}%" if pd.notna(g) else "--"
    gc = "#10b981" if pd.notna(g) and g >= 0 else "#ef4444"
    yoy_html += f'<div class="kpi-card"><div class="kpi-label">{int(row["order_year"])}</div><div class="kpi-value">${row["revenue"]:,.0f}</div><div class="kpi-delta" style="color:{gc}">{gs}</div></div>\n'

# Build data JSON
js_data = {
    "monthly": {"dates": [str(d.date()) for d in mg["ym_date"]], "revenue": mg["revenue"].round(0).tolist(), "profit": mg["profit"].round(0).tolist(), "ma3": mg["ma3"].round(0).tolist(), "margin": mg["margin_pct"].tolist()},
    "cat": {"labels": cat["category"].tolist(), "values": cat["sales"].round(0).tolist()},
    "reg": {"regions": reg["region"].tolist(), "sales": reg["sales"].round(0).tolist(), "profit": reg["profit"].round(0).tolist()},
    "prod": {"names": [n[:45] for n in prod["product_name"]], "sales": prod["sales"].round(0).tolist(), "cats": prod["category"].tolist()},
    "cust": {"names": cust["customer_name"].tolist(), "sales": cust["sales"].round(0).tolist(), "segs": cust["segment"].tolist()},
    "subcat": {"names": subcat["sub_category"].tolist(), "profit": subcat["profit"].round(0).tolist()},
    "heat": {"z": (pivot.values / 1000).round(1).tolist(), "years": [int(y) for y in pivot.index], "months": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]},
    "state": {"states": state["state"].tolist(), "sales": state["sales"].round(0).tolist(), "regions": state["region"].tolist()},
    "disc": {"tiers": disc["discount_tier"].tolist(), "profit": disc["profit"].round(0).tolist(), "loss_rate": disc["loss_rate"].tolist()},
    "scatter": {"names": prod_all["product_name"].tolist(), "cats": prod_all["category"].tolist(), "sales": prod_all["sales"].round(0).tolist(), "profit": prod_all["profit"].round(0).tolist(), "units": prod_all["units"].tolist()},
    "map": {"states": [s[:2].upper() for s in state_map["state"]], "full_names": state_map["state"].tolist(), "sales": state_map["sales"].round(0).tolist(), "profit": state_map["profit"].round(0).tolist(), "regions": state_map["region"].tolist()},
    "custHist": cust_hist
}

# Write template HTML (not an f-string, so JS braces are safe)
print("Writing dashboard.html...")
html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Superstore Sales Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.0.min.js"></script>
<style>
* { margin:0; padding:0; box-sizing:border-box; }
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background:#f0f2f6; }
.header { background:#1b3a5c; color:white; padding:28px 40px; }
.header h1 { font-size:1.7rem; font-weight:700; }
.header p { font-size:0.85rem; color:#aaa; margin-top:4px; }
.container { max-width:1400px; margin:0 auto; padding:20px; }
.row { display:flex; gap:16px; flex-wrap:wrap; margin-bottom:16px; }
.card { background:white; border-radius:12px; padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.06); flex:1; min-width:420px; }
.card-full { background:white; border-radius:12px; padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.06); width:100%; margin-bottom:16px; }
.kpi-row { display:grid; grid-template-columns:repeat(7,1fr); gap:10px; margin-bottom:20px; }
.yoy-row { display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin-bottom:20px; }
.kpi-card { background:#111; border-radius:10px; padding:16px 10px; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.3); }
.kpi-label { font-size:0.7rem; color:#aaa; text-transform:uppercase; letter-spacing:0.5px; margin-bottom:6px; }
.kpi-value { font-size:1.3rem; font-weight:700; color:#fff; }
.kpi-delta { font-size:0.8rem; color:#10b981; margin-top:4px; }
.section-title { font-size:1.2rem; font-weight:700; color:#1b3a5c; margin-bottom:12px; padding-bottom:6px; border-bottom:2px solid #e0e0e0; }
.tab-nav { display:flex; gap:4px; margin-bottom:16px; flex-wrap:wrap; }
.tab-btn { padding:10px 22px; border:none; border-radius:8px; cursor:pointer; font-size:0.9rem; font-weight:600; background:#ddd; color:#555; transition:all 0.2s; }
.tab-btn.active { background:#1b3a5c; color:white; }
.tab-btn:hover { background:#2563eb; color:white; }
.tab-content { display:none; }
.tab-content.active { display:block; }
.chart { width:100%; height:400px; }
.chart-tall { width:100%; height:500px; }
.footer { text-align:center; padding:25px; color:#999; font-size:0.8rem; }
@media (max-width:900px) { .kpi-row{grid-template-columns:repeat(3,1fr);} .yoy-row{grid-template-columns:repeat(2,1fr);} .card{min-width:100%;} }
</style>
</head>
<body>
<div class="header">
    <h1>Superstore Sales Dashboard</h1>
    <p>Performance analytics for FY 2014-2017 | 9,986 transactions | Data: clean_data.csv</p>
</div>
<div class="container">
<div class="tab-nav">
    <button class="tab-btn active" onclick="showTab('overview',this)">Executive Overview</button>
    <button class="tab-btn" onclick="showTab('trends',this)">Sales Trends</button>
    <button class="tab-btn" onclick="showTab('products',this)">Products</button>
    <button class="tab-btn" onclick="showTab('customers',this)">Customers</button>
    <button class="tab-btn" onclick="showTab('geography',this)">Geography</button>
</div>

<div id="overview" class="tab-content active">
<div class="kpi-row">__KPI_CARDS__</div>
<div class="row">
    <div class="card"><div id="c_monthly" class="chart"></div></div>
    <div class="card"><div id="c_category" class="chart"></div></div>
</div>
<div class="row">
    <div class="card"><div id="c_region" class="chart"></div></div>
    <div class="card"><div id="c_discount" class="chart"></div></div>
</div>
</div>

<div id="trends" class="tab-content">
<div class="section-title">Year-over-Year Performance</div>
<div class="yoy-row">__YOY_CARDS__</div>
<div class="card-full"><div id="c_revmargin" class="chart"></div></div>
<div class="card-full"><div id="c_heatmap" class="chart-tall"></div></div>
</div>

<div id="products" class="tab-content">
<div class="row">
    <div class="card"><div id="c_topprods" class="chart-tall"></div></div>
    <div class="card"><div id="c_subcat" class="chart-tall"></div></div>
</div>
<div class="card-full"><div id="c_scatter" class="chart-tall"></div></div>
</div>

<div id="customers" class="tab-content">
<div class="row">
    <div class="card"><div id="c_topcusts" class="chart-tall"></div></div>
    <div class="card"><div id="c_custhist" class="chart-tall"></div></div>
</div>
</div>

<div id="geography" class="tab-content">
<div class="card-full"><div id="c_usmap" class="chart-tall"></div></div>
<div class="card-full"><div id="c_topstates" class="chart-tall"></div></div>
</div>
</div>

<div class="footer">Superstore Sales Dashboard | Data: clean_data.csv (9,986 transactions, 2014-2017) | Built with Plotly</div>

<script>
var D = __DATA_JSON__;
var L = { paper_bgcolor:'rgba(0,0,0,0)', plot_bgcolor:'rgba(0,0,0,0)', margin:{l:50,r:30,t:45,b:40}, font:{family:'Segoe UI, sans-serif'} };
var C = { responsive:true, displayModeBar:false };

function mc(id, data, lay) {
    var merged = Object.assign({}, L, lay);
    Plotly.newPlot(id, data, merged, C);
}

mc('c_monthly', [
    { x:D.monthly.dates, y:D.monthly.revenue, type:'scatter', mode:'lines+markers', name:'Revenue', line:{color:'#2563eb',width:2} },
    { x:D.monthly.dates, y:D.monthly.profit, type:'scatter', mode:'lines+markers', name:'Profit', line:{color:'#10b981',width:2} },
    { x:D.monthly.dates, y:D.monthly.ma3, type:'scatter', mode:'lines', name:'3M Avg', line:{color:'#ef4444',width:1.5,dash:'dash'} }
], { title:'Monthly Revenue & Profit Trend', yaxis:{tickprefix:'$',tickformat:',.0f'} });

mc('c_category', [
    { labels:D.cat.labels, values:D.cat.values, type:'pie', hole:0.55, textinfo:'percent+label', marker:{colors:['#2563eb','#f59e0b','#10b981']} }
], { title:'Revenue by Category', showlegend:false });

mc('c_region', [
    { y:D.reg.regions, x:D.reg.sales, type:'bar', name:'Sales', orientation:'h', marker:{color:'#2563eb'} },
    { y:D.reg.regions, x:D.reg.profit, type:'bar', name:'Profit', orientation:'h', marker:{color:'#10b981'} }
], { title:'Sales vs Profit by Region', barmode:'group', xaxis:{tickprefix:'$',tickformat:',.0f'} });

mc('c_discount', [
    { x:D.disc.tiers, y:D.disc.profit, type:'bar', name:'Profit', marker:{color:D.disc.profit.map(function(v){return v>=0?'#10b981':'#ef4444'})} },
    { x:D.disc.tiers, y:D.disc.loss_rate, type:'scatter', mode:'lines+markers', name:'Loss Rate %', yaxis:'y2', line:{color:'#ef4444',width:2} }
], { title:'Discount Impact on Profitability', yaxis:{title:'Profit ($)',tickprefix:'$',tickformat:',.0f'}, yaxis2:{title:'Loss Rate %',overlaying:'y',side:'right',ticksuffix:'%'} });

mc('c_revmargin', [
    { x:D.monthly.dates, y:D.monthly.revenue, type:'bar', name:'Revenue', marker:{color:'#2563eb',opacity:0.6} },
    { x:D.monthly.dates, y:D.monthly.margin, type:'scatter', mode:'lines+markers', name:'Margin %', yaxis:'y2', line:{color:'#10b981',width:2.5} }
], { title:'Revenue & Profit Margin Over Time', yaxis:{title:'Revenue ($)',tickprefix:'$',tickformat:',.0f'}, yaxis2:{title:'Margin %',overlaying:'y',side:'right',ticksuffix:'%'} });

mc('c_heatmap', [
    { z:D.heat.z, x:D.heat.months, y:D.heat.years, type:'heatmap', colorscale:'YlOrRd', text:D.heat.z.map(function(r){return r.map(function(v){return v+'K'})}), texttemplate:'%{text}', hovertemplate:'Year %{y}, %{x}: $%{text}<extra></extra>' }
], { title:'Monthly Revenue Heatmap ($K)' });

mc('c_topprods', [
    { x:D.prod.sales, y:D.prod.names, type:'bar', orientation:'h', marker:{color:D.prod.cats.map(function(c){return c==='Technology'?'#2563eb':c==='Furniture'?'#f59e0b':'#10b981'})}, text:D.prod.sales.map(function(v){return '$'+v.toLocaleString()}), textposition:'outside' }
], { title:'Top 10 Products by Revenue', yaxis:{autorange:'reversed'}, xaxis:{tickprefix:'$',tickformat:',.0f'} });

mc('c_subcat', [
    { x:D.subcat.profit, y:D.subcat.names, type:'bar', orientation:'h', marker:{color:D.subcat.profit.map(function(v){return v>=0?'#10b981':'#ef4444'})}, text:D.subcat.profit.map(function(v){return '$'+v.toLocaleString()}), textposition:'outside' }
], { title:'Profit by Sub-Category', yaxis:{autorange:'reversed'}, xaxis:{zeroline:true,zerolinecolor:'black',tickprefix:'$',tickformat:',.0f'} });

mc('c_scatter', [
    { x:D.scatter.sales, y:D.scatter.profit, type:'scatter', mode:'markers', text:D.scatter.names, marker:{size:D.scatter.units.map(function(v){return Math.max(Math.sqrt(v)*3,5)}), color:D.scatter.cats.map(function(c){return c==='Technology'?'#2563eb':c==='Furniture'?'#f59e0b':'#10b981'}), opacity:0.6}, hovertemplate:'%{text}<br>Sales: $%{x:,.0f}<br>Profit: $%{y:,.0f}<extra></extra>' }
], { title:'Product Performance Matrix', xaxis:{tickprefix:'$',tickformat:',.0f',title:'Total Sales'}, yaxis:{tickprefix:'$',tickformat:',.0f',title:'Total Profit'} });

mc('c_topcusts', [
    { x:D.cust.sales, y:D.cust.names, type:'bar', orientation:'h', marker:{color:D.cust.segs.map(function(s){return s==='Consumer'?'#2563eb':s==='Corporate'?'#f59e0b':'#10b981'})}, text:D.cust.sales.map(function(v){return '$'+v.toLocaleString()}), textposition:'outside' }
], { title:'Top 10 Customers by Revenue', yaxis:{autorange:'reversed'}, xaxis:{tickprefix:'$',tickformat:',.0f'} });

mc('c_custhist', [
    { x:D.custHist, type:'histogram', nbinsx:40, marker:{color:'#2563eb',opacity:0.7} }
], { title:'Customer Revenue Distribution', xaxis:{title:'Total Revenue ($)',tickprefix:'$',tickformat:',.0f'}, yaxis:{title:'Number of Customers'} });

mc('c_usmap', [
    { type:'choropleth', locations:D.map.states, locationmode:'USA-states', scope:'usa', z:D.map.sales, colorscale:'Blues', text:D.map.full_names, hovertemplate:'%{text}<br>Revenue: $%{z:,.0f}<extra></extra>', colorbar:{tickprefix:'$',tickformat:',.0f'} }
], { title:'Revenue by State', geo:{scope:'usa'} });

mc('c_topstates', [
    { x:D.state.sales, y:D.state.states, type:'bar', orientation:'h', marker:{color:D.state.regions.map(function(r){return r==='West'?'#2563eb':r==='East'?'#f59e0b':r==='Central'?'#10b981':'#8b5cf6'})}, text:D.state.sales.map(function(v){return '$'+v.toLocaleString()}), textposition:'outside' }
], { title:'Top 15 States by Revenue', yaxis:{autorange:'reversed'}, xaxis:{tickprefix:'$',tickformat:',.0f'} });

function showTab(id, btn) {
    document.querySelectorAll('.tab-content').forEach(function(el){el.classList.remove('active')});
    document.querySelectorAll('.tab-btn').forEach(function(el){el.classList.remove('active')});
    document.getElementById(id).classList.add('active');
    btn.classList.add('active');
    setTimeout(function(){ window.dispatchEvent(new Event('resize')); }, 100);
}
</script>
</body>
</html>"""

# Build KPI cards HTML
kpi_cards = f"""
<div class="kpi-card"><div class="kpi-label">Total Sales</div><div class="kpi-value">${total_sales:,.0f}</div></div>
<div class="kpi-card"><div class="kpi-label">Total Profit</div><div class="kpi-value">${total_profit:,.0f}</div><div class="kpi-delta">{margin:.1f}% margin</div></div>
<div class="kpi-card"><div class="kpi-label">Total Orders</div><div class="kpi-value">{total_orders:,}</div></div>
<div class="kpi-card"><div class="kpi-label">Customers</div><div class="kpi-value">{total_customers:,}</div></div>
<div class="kpi-card"><div class="kpi-label">Avg Order Value</div><div class="kpi-value">${avg_order:,.2f}</div></div>
<div class="kpi-card"><div class="kpi-label">Profit Margin</div><div class="kpi-value">{margin:.1f}%</div></div>
<div class="kpi-card"><div class="kpi-label">Loss Transactions</div><div class="kpi-value">{loss_count:,}</div><div class="kpi-delta" style="color:#ef4444">{loss_count/len(df)*100:.1f}% loss rate</div></div>
"""

# Substitute placeholders
html = html.replace("__KPI_CARDS__", kpi_cards.strip())
html = html.replace("__YOY_CARDS__", yoy_html.strip())
html = html.replace("__DATA_JSON__", json.dumps(js_data))

with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(html)

size_mb = os.path.getsize(OUT_PATH) / (1024 * 1024)
print(f"Saved: {OUT_PATH} ({size_mb:.1f} MB)")
print("Open dashboard.html in any browser -- no server needed.")
